# 避難所候補算出支援ツール（sheltermatch）

要支援者一覧（緯度・経度入り）と避難所一覧の座標から、要支援者ごとに **距離が近い避難所候補（既定で上位3件）** を
算出し、必要であればハザード区域との位置関係も確認して、職員の最終判断のための資料（CSV）を作成するNotebookです。

**要支援者一覧CSVの正式フォーマット**: `resident_id,address,latitude,longitude,geocode_status` の
5列です（テンプレート: `templates/residents.csv`）。これは別Notebook `address_geocode.ipynb`
（住所→座標変換）の出力と同じ形式のため、`address_geocode.ipynb` で変換したCSVは、**加工せずそのまま
ここへアップロード**できます。

```text
templates/residents.csv
  ↓
address_geocode.ipynb（住所→座標変換）
  ↓
同じ5列のCSV
  ↓
sheltermatch.ipynb（このNotebook）
```

住所しかない場合は、先に `address_geocode.ipynb` で緯度・経度を付与してください（このNotebook自体は
住所→座標変換は行いません）。5列がすべて揃っていない要支援者CSVは受け付けません。

**このツールが行うこと**
- 直線距離（`geopy.distance.geodesic`）による避難所候補の算出（候補提示。自動割当ではありません）
- 結果を地図上で目視確認するためのレビュー用HTMLの作成（`sheltermatch_review.zip`。正式なデータ
  成果物は `assigned_shelters.csv` で、HTMLはその確認用の補助成果物です）
- 任意機能として、要支援者・避難所・両者を結ぶ直線とハザード区域（GeoJSON / Shapefile。国・県等の
  公式配布ZIPも展開・変換せずそのままアップロード可能）との位置関係の確認

**このツールが行わないこと（重要）**
- 住所→座標変換（要支援者一覧CSVの `latitude` / `longitude` にあらかじめ座標を入力してください。
  住所しかない場合は、別Notebookの `address_geocode.ipynb` で事前に緯度・経度を付与してから、
  その出力CSVをここへアップロードしてください）
- 避難先・避難経路の自動決定、道路経路・通行可能性の計算（直線距離であり道路距離ではありません。
  直線とハザード区域の交差判定も、道路上の避難経路判定ではありません）
- ハザード判定結果による候補避難所の自動除外・自動順位変更、独自の危険度スコアリング

距離順位とハザード判定は別々の情報として出力します。どの避難所を選ぶかは、出力結果を確認した **職員が判断** してください。

**要支援者CSVの座標が使えなかった場合**
- `match_status` = `no_coordinates`（`latitude` / `longitude` が空欄）
- `match_status` = `invalid_coordinates`（値は入っているが、数値として読めない・緯度経度の範囲外）
- どちらの場合も行は削除せず、入力した `latitude` / `longitude` もそのまま結果CSVへ残します
  （どの行をどう直せばよいか、結果CSVだけで分かるようにするためです）

**ハザード判定結果の読み方**
- `True` = ハザード区域内（境界上含む）または交差あり
- `False` = 有効な座標で判定した結果、ハザード区域外
- 空欄（NaN） = 座標が無い・不正、または候補自体が無いなどの理由で **判定できなかった**（区域外という意味ではありません）

**個人情報の取り扱い**
- 実際の要支援者データ・ハザードデータはこのリポジトリにコミットしないでください（サンプルを追加する場合は完全な架空データを使用してください）。
- 出力CSVには住所・座標等の個人情報が含まれ得ます。取り扱いに注意してください。

## 使い方

1. 下の「利用者設定」を確認する
2. 「ランタイム → すべてのセルを実行」
3. 要支援者CSV（`resident_id,address,latitude,longitude,geocode_status` の5列必須。
   `latitude` / `longitude` の値は行ごとに欠損・不正でも構いません。`resident_id` は空欄・重複不可です。
   `address_geocode.ipynb` の出力CSVはそのままアップロードできます）をアップロードする
4. 必要な場合のみハザードデータ（GeoJSON・Shapefile、または国・県等の公式配布ZIP）をアップロードする
5. BODIK Data APIからの避難所取得に失敗した場合のみ、避難所CSVをアップロードする
6. 結果CSV（`assigned_shelters.csv`）と、レビュー用HTML（`sheltermatch_review.zip`）を保存する
   （ZIPを展開して `review.html` を開くと、地図上で候補・ハザードを確認できます）

コードを読み込まなくても、この説明と各セルのprint出力だけで操作できます。


In [ ]:
# ===== 利用者設定 =====
# 通常変更が必要な項目はこれだけです。値を確認・変更してから実行してください。

# 要支援者・避難所とハザード区域（GeoJSON）との位置関係を確認する場合は True にしてください。
ENABLE_HAZARD_CHECK = False

# 要支援者ごとに算出する避難所候補の件数（避難所がこの件数未満の場合は存在する件数まで出力）
TOP_N = 3

# 避難所一覧の取得方法。"api"=BODIK Data APIから取得 / "csv"=CSVファイルをアップロード
# "api"で取得に失敗した場合は、自動的にCSVアップロードへ切り替わります。
SHELTER_SOURCE = "api"

if not isinstance(TOP_N, int) or TOP_N < 1:
    raise ValueError(f"TOP_N は1以上の整数を指定してください。現在の値: {TOP_N!r}")

if SHELTER_SOURCE not in ("api", "csv"):
    raise ValueError(f"SHELTER_SOURCE は 'api' または 'csv' を指定してください。現在の値: {SHELTER_SOURCE!r}")

print("利用者設定を読み込みました。")
print(f"  ENABLE_HAZARD_CHECK = {ENABLE_HAZARD_CHECK}")
print(f"  TOP_N               = {TOP_N}")
print(f"  SHELTER_SOURCE      = '{SHELTER_SOURCE}'")


In [ ]:
# ===== 実行環境準備 =====
# Google Colabに標準で入っていないライブラリをインストールします（初回のみ数十秒かかることがあります）。
%pip install -q geopy geopandas shapely matplotlib

import hashlib
import io
import json
import re
import shutil
import tempfile
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from geopy.distance import geodesic

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")  # 画面を持たない環境でもPNGを書き出せるようにする
import matplotlib.pyplot as plt
from shapely import make_valid, union_all
from shapely.geometry import Point, LineString, box

from google.colab import files

# 工程別の処理時間（秒）を記録する軽量な計測用の入れ物。time.perf_counter()による簡易計測のためだけに
# 追加したもので、既存の処理結果・ロジックは変更しない（詳細なprofilingは行わない）。
TIMINGS = {}

print("ライブラリの読み込みが完了しました。")


In [ ]:
# ===== 要支援者CSV読込・入力チェック =====
# 要支援者一覧CSVを選択してください（ファイル名は自由です）。共通住民CSVの正式フォーマットは
# resident_id,address,latitude,longitude,geocode_status の5列です（address_geocode.ipynbの
# 出力と同じ形式のため、そのままアップロードできます）。5列すべてを必須列として確認し、
# 不足していれば処理を止めます。resident_idは住所変換結果との対応が崩れないよう、空欄・重複が
# あればここでも処理を止めます（address_geocode.ipynbと同じ最低限のチェック）。
# latitude / longitude の値は行ごとに欠損・不正でも構いません（行を削除せず、後続の距離計算のみ
# 対象外とします。match_status 列で no_coordinates（空欄）/ invalid_coordinates（値は入って
# いるが数値として読めない・範囲外）として確認できます）。入力した値は書き換えず、そのまま
# 結果CSVへ残します。
# geocode_status列は変更せずそのまま結果CSVへ保持します（座標→避難所候補の突合結果である
# sheltermatch自身のmatch_status列とは別の情報のため、混同しないでください）。
# 文字コードは UTF-8 (BOM付き) → CP932 → UTF-8 の順に自動判定します。resident_idは
# address_geocode.ipynbと同様、先頭ゼロ等の表記を保つため文字列として読み込みます。

print("【要支援者一覧CSV】を選択してください。")
uploaded_residents = files.upload()

if len(uploaded_residents) == 0:
    raise RuntimeError("要支援者一覧CSVがアップロードされませんでした。ファイルを1つ選択してください。")
if len(uploaded_residents) > 1:
    raise RuntimeError("要支援者一覧CSVは1つだけ選択してください。")

residents_filename = list(uploaded_residents.keys())[0]
residents_bytes = uploaded_residents[residents_filename]
print(f"'{residents_filename}' を要支援者一覧として受け取りました。")

# 要支援者CSVを新たに読み込むたびに、この時点から性能計測をやり直す（TIMINGSをリセットする）。
# ハザードデータ・避難所データを読み込み直さずに、この「要支援者CSV読込・入力チェック」セルから
# 100/500/1000件を切り替えて再実行した場合に、前回実行分のhazard/shelters等の時間が今回の
# 合計へ混入しないようにするための措置（ハザードGeoDataFrame・避難所DataFrame自体は再利用できる。
# 計測用の辞書だけを空にする）。
TIMINGS = {}

_stage_start = time.perf_counter()


def read_csv_auto(file_bytes, label, dtype=None):
    """UTF-8(BOM付き) → CP932 → UTF-8 の順で読み込みを試み、成功したDataFrameを返す。"""
    encodings = [
        ("utf-8-sig", "UTF-8 (BOM付き)"),
        ("cp932", "CP932 (Shift-JIS系)"),
        ("utf-8", "UTF-8"),
    ]
    last_error = None
    for encoding, encoding_label in encodings:
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), encoding=encoding, dtype=dtype)
            print(f"[{label}] {encoding_label} として読み込みました。（{len(df)}行）")
            return df
        except (UnicodeDecodeError, UnicodeError) as error:
            last_error = error
            continue
    raise ValueError(
        f"[{label}] 文字コードを判定できませんでした。"
        "UTF-8(BOM付き)・CP932・UTF-8のいずれでも読み込めません。"
        "Excel等での保存時の文字コードを確認してください。"
        f" 詳細: {last_error}"
    )


def ensure_coordinate_columns(df, label):
    """latitude/longitude列が両方存在することを確認する（値の欠損・不正は許容し、ここでは
    行を削除しない。列自体が無い場合のみ列不足として停止する）。避難所一覧CSV用。"""
    missing = [c for c in ("latitude", "longitude") if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 必須列が見つかりません: {', '.join(missing)}")
    return df


def is_blank(series):
    """CSVの1列について、値が空欄（欠損、または空白だけ）の行をTrueとするSeriesを返す。
    「空欄」と「値は入っているが使えない」を区別するため、複数の入力チェックから共通で使う。"""
    return series.isna() | (series.astype(str).str.strip() == "")


# 共通住民CSVの正式フォーマット（address_geocode.ipynbの入出力と同じ5列）。
RESIDENT_REQUIRED_COLUMNS = ["resident_id", "address", "latitude", "longitude", "geocode_status"]


def ensure_resident_columns(df, label):
    """要支援者一覧CSVが共通住民CSVの5列（resident_id,address,latitude,longitude,geocode_status）
    をすべて持つことを確認する。列そのものは5列とも必須とし、不足していれば処理を止める
    （住所変換input＝output＝sheltermatch inputという共通フォーマットを一本化するため、
    旧来のlatitude/longitudeのみの形式は正式入力として扱わない）。latitude/longitudeの
    セルの値が欠損・不正な行は、従来どおり削除せず許容する。"""
    missing = [c for c in RESIDENT_REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{label}] 必須列が見つかりません: {', '.join(missing)}\n"
            f"共通住民CSVは {','.join(RESIDENT_REQUIRED_COLUMNS)} の5列です。"
            "templates/residents.csv を使用してください。"
        )
    return df


def find_resident_id_problems(df):
    """resident_id列の空欄・重複を検出する。resident_idは住所変換結果と避難所候補算出結果を
    紐づける結合キーのため、曖昧な状態のまま処理を進めない（address_geocode.ipynbと同じチェック）。
    戻り値は問題を説明する文字列のリスト（問題が無ければ空リスト）。行番号はCSV上の行番号
    （1行目をヘッダーとする）で示す。"""
    problems = []
    resident_id = df["resident_id"]

    blank_mask = is_blank(resident_id)
    if blank_mask.any():
        blank_rows = [i + 2 for i in df.index[blank_mask]]
        problems.append(f"resident_id が空欄の行があります（CSV行番号: {', '.join(map(str, blank_rows))}）")

    dup_mask = resident_id.duplicated(keep=False) & ~blank_mask
    if dup_mask.any():
        dup_detail = [f"CSV行{i + 2}='{resident_id.iloc[i]}'" for i in df.index[dup_mask]]
        problems.append(f"resident_id が重複している行があります（{', '.join(dup_detail)}）")

    return problems


def is_valid_coordinate(lat, lon):
    """緯度・経度が数値として有効な範囲かどうかを返す（欠損・範囲外はFalse）。
    距離計算・ハザード判定など、1点ずつ座標を扱う関数から共通で利用する。"""
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)


def read_coordinates(df, label):
    """latitude/longitude列を数値として読み取り、行ごとの状態を判定する。
    戻り値は (緯度Series, 経度Series, 状態Series) で、状態は次の3種類。

      ok                  : 距離計算に使える有効な座標
      no_coordinates      : 緯度・経度のどちらかが空欄
      invalid_coordinates : 値は入っているが、数値として読めない（例:「不明」）か、
                            緯度-90〜90・経度-180〜180の範囲外

    「空欄」と「値は入っているが使えない」を取り違えると、職員がどの行をどう直せばよいか
    分からなくなるため、必ず区別する。元のlatitude/longitude列は書き換えないので、
    入力した値はそのまま結果CSVに残る。"""
    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")

    status = pd.Series("ok", index=df.index)
    status[~(lat.between(-90, 90) & lon.between(-180, 180))] = "invalid_coordinates"
    status[is_blank(df["latitude"]) | is_blank(df["longitude"])] = "no_coordinates"

    unusable_count = int((status != "ok").sum())
    if unusable_count:
        print(f"[{label}] 距離計算に使えない座標の行が {unusable_count}件あります（全{len(df)}行中）。")
    return lat, lon, status


residents = read_csv_auto(residents_bytes, "要支援者一覧", dtype={"resident_id": str})
residents = ensure_resident_columns(residents, "要支援者一覧")

if len(residents) == 0:
    raise ValueError(
        "[要支援者一覧] データ行が1件もありません（見出し行だけのCSVです）。"
        "要支援者の行が入ったCSVをアップロードしてから、再度実行してください。"
    )

resident_id_problems = find_resident_id_problems(residents)
if resident_id_problems:
    raise ValueError(
        "[要支援者一覧] resident_id の内容に問題があるため処理を中止しました。\n"
        + "\n".join(f"  - {problem}" for problem in resident_id_problems)
        + "\nresident_id を空欄・重複のない一意な値に修正してから、再度アップロードしてください。"
    )

resident_lat, resident_lon, resident_coord_status = read_coordinates(residents, "要支援者一覧")

display(residents.head())

TIMINGS["residents_csv"] = time.perf_counter() - _stage_start
print(f"[処理時間] 要支援者CSV読込・入力チェック: {TIMINGS['residents_csv']:.2f}秒")


In [ ]:
# ===== 避難所取得・正規化・入力チェック =====
# 避難所一覧は上の「利用者設定」の SHELTER_SOURCE に従って取得します。
# - "api"（既定）: BODIK Data API（CKANの datastore_search）から自治体標準ODSの避難所データを
#   直接取得します（公開データの読み取りのみのためAPIキーは不要）。取得に失敗した場合
#   （通信エラー・レスポンス異常・0件など）は、Notebookを停止せずCSVアップロードへ自動的に切り替えます。
# - "csv": 避難所一覧CSVをブラウザから選択してアップロードします。
#
# 避難所一覧CSVが自治体標準オープンデータセット(ODS)形式（例:
# https://data.bodik.jp/dataset/472107_evacuation_space ）の場合、日本語列名
# （名称→name、緯度→latitude、経度→longitude）を自動的に内部標準列名へ変換します。
# 従来の name/latitude/longitude 形式のCSVもそのまま利用できます。
# 「災害種別_」で始まる列がある場合は、値の 1(対応済み)/2(2階以上であれば対応済み)/空欄(未対応) の
# 区別を保ったまま、避難所候補ごとにCSVへ参考情報として出力します（距離順位や候補の自動除外には
# 使用しません。これは指定緊急避難場所としての対応種別情報であり、後述のGeoJSONによるハザード判定
# とは別の情報です）。
# 座標が不正な避難所の行は距離計算の対象から除外します。

_stage_start = time.perf_counter()

BODIK_BASE_URL = "https://data.bodik.jp"
BODIK_RESOURCE_ID = "3132a0a4-f522-4b2d-bf18-f106d8b3a5ae"  # 糸満市 指定緊急避難場所データセット


def fetch_shelters_from_bodik(base_url, resource_id, page_size=1000):
    """BODIKのCKAN Data API(datastore_search)から避難所データを全件取得し、DataFrameで返す。
    total/offsetでページングして全件取得する。取得できない場合は例外を発生させ、
    呼び出し側でCSVアップロードへフォールバックする。"""
    endpoint = f"{base_url}/api/action/datastore_search"
    records = []
    offset = 0
    total = None

    while True:
        response = requests.get(
            endpoint,
            params={"resource_id": resource_id, "limit": page_size, "offset": offset},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

        if not payload.get("success"):
            raise RuntimeError("CKAN APIレスポンスが success=false を返しました。")

        result = payload.get("result")
        if result is None or "records" not in result:
            raise RuntimeError("CKAN APIレスポンスに result.records が含まれていません。")

        page_records = result["records"]
        records.extend(page_records)

        if total is None:
            total = result.get("total", len(page_records))

        offset += len(page_records)
        if len(page_records) == 0 or offset >= total:
            break

    if len(records) == 0:
        raise RuntimeError("BODIK APIの取得結果が0件でした。")

    return pd.DataFrame(records)


shelters_raw = None
shelters_bytes = None

if SHELTER_SOURCE == "api":
    try:
        shelters_raw = fetch_shelters_from_bodik(BODIK_BASE_URL, BODIK_RESOURCE_ID)
        print(f"BODIK APIから避難所一覧を{len(shelters_raw)}件取得しました。")
    except Exception as error:
        print("BODIK APIから避難所一覧を取得できませんでした。")
        print("CSVファイルから読み込みます。")
        print(f"（詳細: {error}）")

if shelters_raw is None:
    print("【避難所一覧CSV】を選択してください。")
    uploaded_shelters = files.upload()

    if len(uploaded_shelters) == 0:
        raise RuntimeError("避難所一覧CSVがアップロードされませんでした。ファイルを1つ選択してください。")
    if len(uploaded_shelters) > 1:
        raise RuntimeError("避難所一覧CSVは1つだけ選択してください。")

    shelters_filename = list(uploaded_shelters.keys())[0]
    shelters_bytes = uploaded_shelters[shelters_filename]
    print(f"'{shelters_filename}' を避難所一覧として受け取りました。")

if shelters_bytes is not None:
    shelters_raw = read_csv_auto(shelters_bytes, "避難所一覧")

# 避難所一覧の列名アイリアス（自治体標準ODS等の日本語列名 → 内部標準列名）
SHELTER_COLUMN_ALIASES = {
    "name": ["名称"],
    "latitude": ["緯度"],
    "longitude": ["経度"],
}

# 避難所の災害種別列の接頭辞（この接頭辞で始まる列を動的にすべて認識する）
DISASTER_TYPE_COLUMN_PREFIX = "災害種別_"

# 自治体標準ODSの災害種別列の値と対応区分の対応表（BODIKの指定緊急避難場所データセット仕様）
# 1=対応済み、2=2階以上であれば対応済み、空欄=未対応。これ以外の値は独自に「対応」と推測しない。
DISASTER_SUPPORT_LABELS = {
    "1": "対応済み",
    "2": "2階以上であれば対応済み",
}


def normalize_code_value(value):
    """公式データのコード値を、対応表を引くための文字列へ揃える。同じコードでも読込方法によって
    1 / "1" / 1.0 のように型や表現が変わるため、整数として解釈できる場合は整数の文字列表現
    （"1"）に統一する。欠損値はNoneを返す。1.5のような整数でない値を丸めると既知コードへ誤分類
    される恐れがあるため丸めず、元の値の文字列表現のまま返す（未知のコード値も消さずに残す）。
    避難所の災害種別列と、ハザードデータのコード属性（A33等）の両方から使う。"""
    if pd.isna(value):
        return None

    text = str(value).strip()
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except (TypeError, ValueError):
        pass
    return text


def normalize_shelter_columns(df, label):
    """自治体標準ODS等で使われる日本語列名を、内部標準列名(name/latitude/longitude)へ変換する。
    既に標準列名がある場合はそちらを優先し、変換しない。使用した列名の対応表も返す。"""
    df = df.copy()
    used_columns = {}
    for standard_col, aliases in SHELTER_COLUMN_ALIASES.items():
        if standard_col in df.columns:
            used_columns[standard_col] = standard_col
            continue
        for alias in aliases:
            if alias in df.columns:
                df = df.rename(columns={alias: standard_col})
                used_columns[standard_col] = alias
                break

    renamed = {std: orig for std, orig in used_columns.items() if orig != std}
    if renamed:
        mapping_text = ", ".join(f"{orig}→{std}" for std, orig in renamed.items())
        print(f"[{label}] 自治体標準ODS等の日本語列名を自動変換しました: {mapping_text}")

    return df, used_columns


def detect_disaster_type_columns(df):
    """列名が '災害種別_' で始まる列を、対応する災害種別情報として動的に検出する。
    特定の災害種別名の一覧に固定せず、実データに存在する列をそのまま採用する。"""
    return [c for c in df.columns if c.startswith(DISASTER_TYPE_COLUMN_PREFIX)]


def classify_disaster_support(value):
    """自治体標準ODSの災害種別列の値を解釈し、対応区分のラベルを返す（未対応ならNone）。
    '1'=対応済み、'2'=2階以上であれば対応済み、空欄=未対応という標準仕様に従う。
    それ以外の想定外の値は独自に「対応している」と推測せず、値をそのまま保持して要確認として返す。"""
    code = normalize_code_value(value)
    if code is None or code == "":
        return None
    if code in DISASTER_SUPPORT_LABELS:
        return DISASTER_SUPPORT_LABELS[code]
    return f"{code}(要確認)"


def build_disaster_support(df, disaster_columns):
    """行ごとに対応している災害種別とその対応区分を ';' 区切りでまとめたSeriesを返す。
    例: '洪水:対応済み;高潮:2階以上であれば対応済み'。未対応(空欄)の種別は含めない。
    災害種別列が無ければ全行NaN。"""
    if not disaster_columns or len(df) == 0:
        return pd.Series([np.nan] * len(df), index=df.index, dtype="object")

    prefix_len = len(DISASTER_TYPE_COLUMN_PREFIX)

    def row_support(row):
        parts = []
        for col in disaster_columns:
            label = classify_disaster_support(row[col])
            if label is not None:
                parts.append(f"{col[prefix_len:]}:{label}")
        return ";".join(parts)

    return df.apply(row_support, axis=1)


# 避難所一覧は座標列の存否を確認する前に、自治体標準ODS等の日本語列名を内部標準列名へ変換する
shelters, shelter_used_columns = normalize_shelter_columns(shelters_raw, "避難所一覧")
shelter_disaster_columns = detect_disaster_type_columns(shelters)
shelters["_disaster_support"] = build_disaster_support(shelters, shelter_disaster_columns)
HAS_DISASTER_TYPE_COLUMNS = len(shelter_disaster_columns) > 0

shelters = ensure_coordinate_columns(shelters, "避難所一覧")
if "name" not in shelters.columns:
    raise ValueError("[避難所一覧] 名称列が見つかりません: name または 名称 の列が必要です。")

shelters["latitude"], shelters["longitude"], shelter_coord_status = read_coordinates(
    shelters, "避難所一覧"
)

# 座標が不正な避難所の行は距離計算の対象から除外する（有効な避難所が0件の場合は処理を停止する）
shelters_valid = shelters.loc[shelter_coord_status == "ok"].reset_index(drop=True)
invalid_shelter_count = len(shelters) - len(shelters_valid)

if len(shelters_valid) == 0:
    raise RuntimeError(
        "有効な座標を持つ避難所が0件です。避難所一覧CSVの latitude / longitude 列を確認してください。"
    )

print(
    f"[避難所一覧] 避難所件数: {len(shelters)}件 / "
    f"名称列: '{shelter_used_columns.get('name', 'name')}' / "
    f"緯度列: '{shelter_used_columns.get('latitude', 'latitude')}' / "
    f"経度列: '{shelter_used_columns.get('longitude', 'longitude')}' / "
    f"認識した災害種別列数: {len(shelter_disaster_columns)}件"
)
print(f"距離計算に使用する有効な避難所: {len(shelters_valid)}件（座標不正のため{invalid_shelter_count}件を除外）")

display(shelters.head())

TIMINGS["shelters"] = time.perf_counter() - _stage_start
print(f"[処理時間] 避難所データ取得・整形: {TIMINGS['shelters']:.2f}秒")


## ハザードデータ読込（任意）

`ENABLE_HAZARD_CHECK = True`（上の「利用者設定」）の場合のみ、ハザード区域のデータをアップロードします。
1ファイルにつき **`.geojson` 単体**、または **国・県等の公式配布ZIP（`.zip`）** のいずれかを選択できます。
ZIPの場合、内部のディレクトリ構造やファイル数に関わらず、配下の `.geojson` と **Shapefile**（`.shp`
とその組の `.shx`/`.dbf`。`.prj` は無くても構いません）を自動的に再帰探索してすべて統合します
（利用者が展開・変換・整理する必要はありません）。ZIP内のファイル名がCP932（Shift-JIS系）でUTF-8
フラグを立てずに格納されている場合（沖縄県公式データ等でよく見られます）も、自動的に文字化けを復元
してから展開します。`.shx`/`.dbf` が揃っていないShapefileは、そのレイヤーのみ読み込めない旨をまとめて
表示し、ほかに有効なレイヤーがあれば処理を継続します。区域判定に使えるPolygon/MultiPolygon以外の
ジオメトリ（LineString等）が含まれる場合は、独自に面へ変換したりはせず区域判定対象外として除外します。

公式配布データには、自己交差等で `is_valid=False` になっているPolygon/MultiPolygonが実際に含まれて
います（国土数値情報A31aと沖縄県津波浸水想定の実データで確認済み）。そのまま除外するとその区域が
判定から抜け落ちるため、**`is_valid=False` のPolygon/MultiPolygonに限り `shapely.make_valid()` で
直してから使います**。

- すでに有効なPolygon/MultiPolygonには `make_valid()` をかけず、一切変更しません
- 直した結果がGeometryCollection（面と線が混ざったもの）になる場合は、Polygon/MultiPolygon成分
  だけを使います（線・点の成分は区域ではないため使いません）
- 区域として使える形にならなかったものは、従来どおり除外します
- 空・欠損のジオメトリは、従来どおり区域判定対象外です
- 直した行も、元の `hazard_type`（A31aの詳細カテゴリ・A33の現象種類/区域区分・津波の浸水深区分・
  高潮のカテゴリ）をそのまま引き継ぎます。種別・カテゴリを再解釈することはありません

読込時には、不正なジオメトリが何件あり、そのうち何件を修復して採用し、何件を除外したのかを
まとめて表示します。

アップロードした **ファイル（GeoJSONまたはZIP）ごとに1回だけ** ハザード種別名を決定します（ZIP内の個々の
レイヤーごとには入力しません）。ファイル名が以下のような既知の公式配布形式に一致する場合は、入力を求めず
自動判定します（一致しない場合のみ、これまでどおり種別名の入力を求めます。空欄の場合はファイル名を
`hazard_type` とする挙動も維持されます）。

- 国土数値情報「洪水浸水想定区域データ」（`A31a-` で始まる形式。例: `A31a-25_47_10_GEOJSON.zip`）
  → 河川区分から「洪水（洪水予報河川・水位周知河川）」/「洪水（その他の河川）」を判定
- 国土数値情報「土砂災害警戒区域データ」（`A33-` で始まる形式。例: `A33-25_47_GEOJSON.zip`）→「土砂災害」
  （属性 `A33_001`/`A33_002` があれば現象の種類・区域区分も反映）
- 沖縄県津波浸水想定データ（`level1`〜`level7` で始まる形式。例: `level1_1cm-30cm.zip`）→「津波」
- 沖縄県高潮浸水想定データ（`<市町村コード>_takasiosinnsuisoutei_...` 形式。
  例: `47007_takasiosinnsuisoutei_22itoman.zip`）→「高潮」

ZIPの展開先直下にサブフォルダがある場合（国土数値情報等で「計画規模」「想定最大規模」のようにカテゴリ別に
ファイルが分かれている場合）は、そのサブフォルダ名を **カテゴリ** として扱い、`基本種別名:カテゴリ名` の形で
`hazard_type` に保持します。ただし高潮データは、複数のShapefile（`最大浸水深_糸満市.shp` /
`浸水継続時間_糸満市.shp` 等）が1つのラッパーフォルダにまとめて配布され、フォルダ名だけではレイヤーを
区別できないため、ラッパーフォルダの有無に関わらず、ファイル名から市町村名部分を除いた名前を優先して
カテゴリとします（例: `高潮:最大浸水深` / `高潮:浸水継続時間`）。また、属性に `分類` 列があるレイヤー
（津波浸水想定データ等）は、ポリゴンごとに `分類` の値をカテゴリとして反映します（例:
`津波:0.01m以上0.3m未満`）。土砂災害（A33）は `A33_001`（現象の種類）/`A33_002`（区域区分）の
公式コードリストの名称をそのまま用い、ポリゴンごとに `土砂災害:急傾斜地の崩壊:土砂災害警戒区域(指定済)`
のように反映します（未知のコード値が来た場合も削除せず、コード値自体を保持します）。フォルダ名・
ファイル名・属性の意味は独自解釈せずそのまま使うため、特定の配布元の命名規則以外には依存しません。

座標系はCRS情報がある場合はEPSG:4326へ変換します。CRS情報が無いGeoJSONは既定でWGS84として扱いますが、
CRS情報が無いShapefileについては無条件にEPSG:4326と決めつけず、座標値が経緯度として妥当な範囲
（経度-180〜180・緯度-90〜90）に収まる場合のみEPSG:4326とみなし、収まらない場合はそのレイヤーを座標系
不明として除外します。いずれの形式でもPolygon/MultiPolygon以外の空・不正なジオメトリは除外されます。
`ENABLE_HAZARD_CHECK = False`（既定）の場合はこのセルはスキップされ、ハザードデータなしで距離候補算出のみ
が実行されます。

**注意**: `ENABLE_HAZARD_CHECK = True` にした場合、有効なハザード区域ポリゴンが1件も読み込めなかったときは
「ハザードなし」とみなさず、ここで処理を停止します（判定していないことと、ハザード区域でないことを区別するためです）。


In [ ]:
def _decode_zip_entry_name(member):
    """ZIPエントリのファイル名を復号する。UTF-8フラグ（汎用目的ビットフラグのbit 11）が
    立っていないエントリは、Pythonのzipfileが既定でCP437として解釈するため、CP932
    (Shift-JIS系)の日本語ファイル名を含む配布ZIP（沖縄県公式データ等、UTF-8フラグを立てずに
    作成されたZIP）では文字化けする。その場合は元のバイト列をCP932として再解釈する
    （変換できない場合は元の文字列のまま扱う）。"""
    if member.flag_bits & 0x800:
        return member.filename
    try:
        return member.filename.encode("cp437").decode("cp932")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return member.filename


def _normalize_zip_entry_separators(name):
    """ZIPエントリ名のパス区切りを '/'（ZIP標準の区切り）に正規化する。国土数値情報A31a等、
    一部の公式配布ZIPはディレクトリ区切りに '\\'（バックスラッシュ）を使って格納されているが、
    '\\' はPOSIX環境のPathlibでは区切り文字として扱われずサブフォルダを認識できないため、
    展開前に '/' へ揃える。'/' はWindows・POSIXどちらのPathlibでも区切り文字として扱われるため、
    実行環境（Colab/Linux・Windows）に関わらず同じディレクトリ構造になる。"""
    return name.replace("\\", "/")


def safe_extract_zip(zip_bytes, extract_dir):
    """ZIPをextract_dirへ安全に展開する。各エントリ名は、文字コード復元
    （_decode_zip_entry_name）→パス区切り正規化（_normalize_zip_entry_separators）の順で
    処理してから扱う。絶対パスや'..'を含むなど、正規化後のパスが展開先ディレクトリの外に出る
    エントリが1件でもあれば、展開を一切行わずに例外を送出する（パストラバーサル対策。
    全エントリを先に検査してから展開する）。"""
    extract_dir_abs = Path(extract_dir).resolve()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        resolved_members = []
        for member in zip_file.infolist():
            name = _decode_zip_entry_name(member)
            name = _normalize_zip_entry_separators(name)
            is_dir_entry = name.endswith("/")
            member_path = (extract_dir_abs / name).resolve()
            if member_path != extract_dir_abs and extract_dir_abs not in member_path.parents:
                raise RuntimeError(
                    "ZIP内に不正なパスが含まれているため展開を中止しました"
                    f"（パストラバーサルの可能性）: {name}"
                )
            resolved_members.append((member, member_path, is_dir_entry))

        for member, member_path, is_dir_entry in resolved_members:
            if is_dir_entry:
                member_path.mkdir(parents=True, exist_ok=True)
                continue
            member_path.parent.mkdir(parents=True, exist_ok=True)
            with zip_file.open(member) as source, open(member_path, "wb") as target:
                target.write(source.read())


def find_shapefile_layers(extract_dir):
    """extract_dir配下の.shpを再帰的に探索し、同じディレクトリに.shx/.dbfが揃っているものだけを
    有効なShapefileレイヤーとして返す（.prjが無くても読み込みは試みる。CRSが無いものとして扱う）。
    戻り値は (有効な.shpパスのリスト, .shx/.dbfが揃っていないため除外した件数, 発見した.shp総数)。"""
    shp_paths = sorted(extract_dir.rglob("*.shp"))
    valid_paths = []
    missing_companion_count = 0
    for shp_path in shp_paths:
        siblings = {p.name.lower() for p in shp_path.parent.iterdir()}
        stem_lower = shp_path.stem.lower()
        if f"{stem_lower}.shx" in siblings and f"{stem_lower}.dbf" in siblings:
            valid_paths.append(shp_path)
        else:
            missing_companion_count += 1
    return valid_paths, missing_companion_count, len(shp_paths)


# 国土数値情報A33（土砂災害警戒区域データ）の公式コードリスト。
# A33_001=現象の種類、A33_002=区域区分。名称は公式コードリストのままとし、独自の呼称（イエロー/
# レッド等）へは置き換えない。
A33_PHENOMENON_LABELS = {
    "1": "急傾斜地の崩壊",
    "2": "土石流",
    "3": "地滑り",
}
A33_ZONE_LABELS = {
    "1": "土砂災害警戒区域(指定済)",
    "2": "土砂災害特別警戒区域(指定済)",
    "3": "土砂災害警戒区域(指定前)",
    "4": "土砂災害特別警戒区域(指定前)",
}


def derive_feature_hazard_types(gdf, hazard_type):
    """レイヤーの属性から、行（ポリゴン）ごとのhazard_typeを組み立てる。国土数値情報A33
    （土砂災害警戒区域データ）の 'A33_001'（現象の種類）/'A33_002'（区域区分）属性がある場合は
    公式コードリストの名称をそのまま用いて '<hazard_type>:<現象の種類>:<区域区分>' とする
    （未知のコード値は削除せず、コード値自体をそのまま使う）。津波浸水想定データ等の『分類』属性が
    ある場合は、従来どおり '<hazard_type>:<分類の値>' とする。どちらも無い場合はhazard_typeを
    全行へそのまま適用する。データセットが増えてもこの関数の中だけで判定し、呼び出し側を
    巨大なif文にしない。"""
    if "A33_001" in gdf.columns and "A33_002" in gdf.columns:
        def build_a33_hazard_type(row):
            phenomenon_code = normalize_code_value(row["A33_001"])
            zone_code = normalize_code_value(row["A33_002"])
            phenomenon = A33_PHENOMENON_LABELS.get(phenomenon_code, phenomenon_code)
            zone = A33_ZONE_LABELS.get(zone_code, zone_code)
            parts = [part for part in (phenomenon, zone) if part]
            return f"{hazard_type}:{':'.join(parts)}" if parts else hazard_type

        return gdf.apply(build_a33_hazard_type, axis=1)

    if "分類" in gdf.columns:
        return gdf["分類"].apply(
            lambda value: f"{hazard_type}:{value}" if pd.notna(value) and str(value).strip() else hazard_type
        )

    return hazard_type


# 区域判定に使えるジオメトリ種別。
POLYGON_TYPES = ["Polygon", "MultiPolygon"]


def empty_hazard_layer():
    """有効なハザード区域が1件も無い場合に返す、空のレイヤー。"""
    return gpd.GeoDataFrame({"hazard_type": [], "geometry": []}, crs="EPSG:4326")


def repair_invalid_polygon(geometry):
    """自己交差等で is_valid=False になっているPolygon/MultiPolygonを、shapely.make_valid()で
    区域判定に使える形へ直す。公式配布データには自己交差を含むポリゴンが実際に含まれており
    （国土数値情報A31a・沖縄県津波浸水想定で確認）、そのまま除外するとその区域が判定から
    抜け落ちるため、ここで直してから使う。

    make_valid()の結果がGeometryCollection（面と線が混ざったもの）になる場合は、
    Polygon/MultiPolygon成分だけを取り出す（線・点の成分は区域ではないため使わない）。
    区域として使える形にならなかった場合はNoneを返し、呼び出し側で従来どおり除外する。

    戻り値は (直したジオメトリ or None, GeometryCollectionから成分を取り出したか)。
    既に有効なジオメトリへは適用しないこと（この関数は呼び出し側でis_valid=Falseの行にだけ使う）。"""
    try:
        repaired = make_valid(geometry)
    except Exception:
        # make_valid自体が失敗した場合も、処理を止めずに従来どおり除外扱いにする
        return None, False

    if repaired is None or repaired.is_empty:
        return None, False

    from_collection = False
    if repaired.geom_type == "GeometryCollection":
        polygon_parts = [
            part for part in repaired.geoms
            if part.geom_type in POLYGON_TYPES and not part.is_empty
        ]
        if not polygon_parts:
            return None, False
        repaired = union_all(polygon_parts)
        from_collection = True

    if repaired.geom_type not in POLYGON_TYPES or repaired.is_empty or not repaired.is_valid:
        return None, False
    return repaired, from_collection


def load_hazard_layer(source, hazard_type, assume_wgs84_without_crs):
    """1件の空間データ（GeoJSONまたはShapefile）を読み込み、hazard_type/geometryの2列に正規化し、
    EPSG:4326へ統一する。区域判定はPolygon/MultiPolygonのみを対象とするため、次のように扱う。

    * 空・欠損のジオメトリ: 区域判定対象外として除外する
    * 有効なPolygon/MultiPolygon: 一切変更せずそのまま使う（make_valid()もかけない）
    * is_valid=FalseのPolygon/MultiPolygon: repair_invalid_polygonで直してから使う
      （直せなかったものは従来どおり除外する）
    * Polygon/MultiPolygon以外（LineString等）: 独自に面へ変換したりはせず除外する

    行ごとのhazard_typeはderive_feature_hazard_typesで組み立てる（『分類』属性やA33の公式属性が
    あれば詳細区分を反映し、無ければhazard_typeをそのまま使う）。ジオメトリを直した行も、
    元のhazard_typeをそのまま引き継ぐ（種別・カテゴリの再解釈はしない）。

    CRSが取得できる場合はEPSG:4326へ変換する。CRSが取得できない場合、assume_wgs84_without_crsが
    Trueなら（GeoJSON等、既定でWGS84として扱われる形式）そのままEPSG:4326とみなす。Falseの場合
    （Shapefile等）は無条件にEPSG:4326と決めつけず、geometry全体のboundsが経緯度として妥当な範囲
    （経度-180~180・緯度-90~90）に収まる場合のみEPSG:4326と推定し、収まらない場合は座標系を判断
    できないとみなして空のGeoDataFrameを返す（呼び出し側で除外扱いにする）。

    戻り値は (GeoDataFrame, crs_note, counts)。crs_noteは 'assumed_by_bounds' / 'unresolved' /
    None。countsは件数の内訳（null_or_empty / non_polygon / repaired /
    repaired_from_collection / unrepairable）で、利用者への集約表示に使う。"""
    counts = Counter()
    gdf = gpd.read_file(source)

    crs_note = None
    if gdf.crs is not None:
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
    elif assume_wgs84_without_crs or len(gdf) == 0:
        gdf = gdf.set_crs(epsg=4326)
    else:
        minx, miny, maxx, maxy = gdf.total_bounds
        if -180 <= minx and maxx <= 180 and -90 <= miny and maxy <= 90:
            gdf = gdf.set_crs(epsg=4326)
            crs_note = "assumed_by_bounds"
        else:
            return empty_hazard_layer(), "unresolved", counts

    geometry = gdf.geometry
    is_null_or_empty = geometry.isna() | geometry.is_empty
    is_polygonal = geometry.geom_type.isin(POLYGON_TYPES)

    usable_mask = ((~is_null_or_empty) & is_polygonal & geometry.is_valid).to_numpy()
    repair_mask = ((~is_null_or_empty) & is_polygonal & (~geometry.is_valid)).to_numpy()

    counts["null_or_empty"] = int(is_null_or_empty.sum())
    counts["non_polygon"] = int(((~is_null_or_empty) & (~is_polygonal)).sum())

    # 有効なジオメトリはそのまま、is_valid=Falseのものだけ直して追加する。
    kept_positions = list(np.flatnonzero(usable_mask))
    kept_geometries = list(geometry.to_numpy()[usable_mask])

    for position in np.flatnonzero(repair_mask):
        repaired, from_collection = repair_invalid_polygon(geometry.iat[position])
        if repaired is None:
            counts["unrepairable"] += 1
            continue
        kept_positions.append(position)
        kept_geometries.append(repaired)
        counts["repaired"] += 1
        if from_collection:
            counts["repaired_from_collection"] += 1

    if not kept_positions:
        return empty_hazard_layer(), crs_note, counts

    kept_rows = gdf.iloc[kept_positions].reset_index(drop=True)
    layer = gpd.GeoDataFrame(
        {
            "hazard_type": derive_feature_hazard_types(kept_rows, hazard_type),
            "geometry": kept_geometries,
        },
        crs="EPSG:4326",
    )
    return layer, crs_note, counts


def _takashio_category_from_stem(stem):
    """高潮データ（例: 最大浸水深_糸満市.shp）のファイル名から、末尾の市町村名部分を除いた
    カテゴリ名を返す（例: '最大浸水深_糸満市' → '最大浸水深'）。区切りが無ければそのまま返す。"""
    prefix, sep, _rest = stem.rpartition("_")
    return prefix if sep else stem


def resolve_layer_hazard_type(base_hazard_type, vector_path, extract_dir):
    """ハザードのカテゴリをhazard_typeへ反映する。国土数値情報等では洪水の中でも「計画規模」
    「想定最大規模」等がZIP展開先直下のサブフォルダで分かれて配布されるため、そのサブフォルダ名を
    そのままカテゴリ名として使う（フォルダ名の意味は独自解釈せず、特定の配布元のディレクトリ名には
    依存しない）。ただし高潮（沖縄県高潮浸水想定データ）は複数のShapefileレイヤーが1つのラッパー
    フォルダにまとめて配布され、フォルダ名だけではレイヤーを区別できないため、フォルダの有無に
    関わらずファイル名から市町村名部分を除いたカテゴリ（例: 最大浸水深/浸水継続時間）を優先して
    用いる。サブフォルダも高潮の命名規則も無い場合はカテゴリなし（基本種別名のみ）とする。"""
    if base_hazard_type == "高潮":
        category = _takashio_category_from_stem(vector_path.stem)
    else:
        rel_parts = vector_path.relative_to(extract_dir).parts
        category = rel_parts[0] if len(rel_parts) > 1 else None
    return f"{base_hazard_type}:{category}" if category else base_hazard_type


def _detect_a31a_hazard_type(filename):
    """国土数値情報A31a（洪水浸水想定区域データ）のファイル名規則
    （例: A31a-25_47_10_GEOJSON.zip）から、年度・都道府県コードには依存せず、
    河川区分（10/20）のみで洪水の種別名を判定する。一致しなければNoneを返す。"""
    match = re.match(r"^A31a-\d+_\d+_(10|20)(?=[_.])", filename)
    if not match:
        return None
    river_classification_labels = {
        "10": "洪水（洪水予報河川・水位周知河川）",
        "20": "洪水（その他の河川）",
    }
    return river_classification_labels[match.group(1)]


def _detect_a33_hazard_type(filename):
    """国土数値情報A33（土砂災害警戒区域データ）のファイル名規則
    （例: A33-25_47_GEOJSON.zip）から、年度・都道府県コードには依存せず土砂災害と判定する。
    ファイル名の先頭部分だけを見るため、Colab等がファイル名重複を避けて末尾に付与する
    '(1)' 等の連番があっても判定できる。一致しなければNoneを返す。"""
    if re.match(r"^A33-\d+_\d+_GEOJSON", filename):
        return "土砂災害"
    return None


def _detect_tsunami_level_hazard_type(filename):
    """沖縄県津波浸水想定データのファイル名規則（例: level1_1cm-30cm.zip、level1～level7）から
    津波と判定する。浸水深の分類はファイル名のlevel番号からは推測せず、実データのShapefile内
    『分類』属性（load_hazard_layerで反映）を優先する。"""
    if re.match(r"^level[1-7](?=[_.])", filename, re.IGNORECASE):
        return "津波"
    return None


def _detect_takashio_hazard_type(filename):
    """沖縄県高潮浸水想定データのファイル名規則
    （例: 47007_takasiosinnsuisoutei_22itoman.zip）から高潮と判定する。"""
    if re.match(r"^\d+_takasiosinnsuisoutei_", filename):
        return "高潮"
    return None


# 既知の公式データ配布ファイル名パターンの判定関数一覧。今後、新しい公式データにも対応する場合は
# ここに判定関数を追加すればよく、巨大なif文へは積み上げない（現時点ではA31a・A33・
# 津波(level1~7)・高潮(takasiosinnsuisoutei)のみ、実データで確認できたファイル名規則として
# 実装している）。
KNOWN_HAZARD_FILENAME_DETECTORS = [
    _detect_a31a_hazard_type,
    _detect_a33_hazard_type,
    _detect_tsunami_level_hazard_type,
    _detect_takashio_hazard_type,
]


def detect_hazard_type(filename):
    """既知の公式データ配布ファイル名パターンからハザード種別名を自動判定する。ZIP・GeoJSON単体の
    どちらのファイル名にも同じ規則を適用する（形式ごとに判定ロジックを二重実装しない）。曖昧な推測は
    せず、既知のパターンのいずれにも一致しない場合はNoneを返す（呼び出し側で利用者入力へフォールバック
    する）。"""
    for detector in KNOWN_HAZARD_FILENAME_DETECTORS:
        hazard_type = detector(filename)
        if hazard_type is not None:
            return hazard_type
    return None


def load_hazard_upload(filename, file_bytes, hazard_type):
    """1つのアップロード（.geojson または国・県等の公式配布ZIP）から、GeoDataFrameを組み立てる。
    ZIPの場合は安全に展開し、内部のディレクトリ構造に関わらず配下の.geojsonと.shp（Shapefile。
    同名の.shx/.dbfが揃っているものに限る）を再帰的に探索する（特定の配布元のファイル名・
    ディレクトリ名には依存しない）。サブフォルダがあればカテゴリとしてhazard_typeに反映し、
    サブフォルダがなければ渡された基本種別名をそのままhazard_typeとする（高潮のみファイル名から
    カテゴリを補う。resolve_layer_hazard_type参照）。利用者には集約したサマリのみ表示し、
    ファイルごとの詳細ログは出さない。"""
    suffix = Path(filename).suffix.lower()
    if suffix not in (".geojson", ".zip"):
        raise ValueError(f"'{filename}' は対応していない形式です。.geojson または .zip を選択してください。")

    layers = []
    empty_file_count = 0
    unresolved_crs_count = 0
    assumed_by_bounds_count = 0
    missing_shapefile_companion_count = 0
    geometry_counts = Counter()

    if suffix == ".geojson":
        layer, crs_note, layer_counts = load_hazard_layer(
            io.BytesIO(file_bytes), hazard_type, assume_wgs84_without_crs=True
        )
        geometry_counts.update(layer_counts)
        if len(layer) == 0:
            empty_file_count += 1
        else:
            layers.append(layer)
    else:
        with tempfile.TemporaryDirectory(prefix="hazard_zip_") as extract_dir_str:
            extract_dir = Path(extract_dir_str)
            safe_extract_zip(file_bytes, extract_dir)
            print(f"'{filename}' を展開しました。")

            geojson_paths = sorted(extract_dir.rglob("*.geojson"))
            shp_paths, missing_shapefile_companion_count, total_shp_found = find_shapefile_layers(extract_dir)

            if not geojson_paths and total_shp_found == 0:
                raise RuntimeError(
                    f"'{filename}' 内に.geojsonまたはShapefile(.shp)が見つかりませんでした。"
                )

            print(f"Shapefileを {len(shp_paths)}レイヤー検出しました。")
            print(f"GeoJSONを {len(geojson_paths)}ファイル検出しました。")

            all_vector_paths = [(p, True) for p in geojson_paths] + [(p, False) for p in shp_paths]

            categories = sorted(
                {
                    p.relative_to(extract_dir).parts[0]
                    for p, _ in all_vector_paths
                    if len(p.relative_to(extract_dir).parts) > 1
                }
            )
            if categories:
                print(f"{len(categories)}カテゴリ（サブフォルダ）を検出しました。")

            for vector_path, is_geojson in all_vector_paths:
                file_hazard_type = resolve_layer_hazard_type(hazard_type, vector_path, extract_dir)
                layer, crs_note, layer_counts = load_hazard_layer(
                    vector_path, file_hazard_type, assume_wgs84_without_crs=is_geojson
                )
                geometry_counts.update(layer_counts)
                if crs_note == "unresolved":
                    unresolved_crs_count += 1
                    continue
                if crs_note == "assumed_by_bounds":
                    assumed_by_bounds_count += 1
                if len(layer) == 0:
                    empty_file_count += 1
                else:
                    layers.append(layer)

    if missing_shapefile_companion_count:
        print(
            f"{missing_shapefile_companion_count}件のShapefileは.shx/.dbfが揃っていないため"
            "読み込めませんでした。"
        )
    if assumed_by_bounds_count:
        print(
            f"{assumed_by_bounds_count}件はCRS情報が無いため、座標値から経緯度データと判断して"
            "EPSG:4326として読み込みました。"
        )
    invalid_polygon_count = geometry_counts["repaired"] + geometry_counts["unrepairable"]
    if invalid_polygon_count:
        print(f"不正なジオメトリ（自己交差等）: {invalid_polygon_count}件")
        print(f"  make_validで修復して採用: {geometry_counts['repaired']}件")
        if geometry_counts["repaired_from_collection"]:
            print(
                "    うちGeometryCollectionからPolygon成分を採用: "
                f"{geometry_counts['repaired_from_collection']}件"
            )
        print(f"  修復できず除外: {geometry_counts['unrepairable']}件")
    if geometry_counts["null_or_empty"]:
        print(
            f"{geometry_counts['null_or_empty']}件は空・欠損のジオメトリのため区域判定対象外です。"
        )
    if geometry_counts["non_polygon"]:
        print(
            f"{geometry_counts['non_polygon']}件はPolygon/MultiPolygon以外のため除外しました"
            "（この区域はハザード判定に含まれません）。"
        )
    if empty_file_count:
        print(f"{empty_file_count}ファイルは有効なPolygon/MultiPolygonを含まないため除外しました。")
    if unresolved_crs_count:
        print(
            f"{unresolved_crs_count}件は座標系を特定できないため除外しました"
            "（CRS情報が無く、座標値も経緯度として妥当な範囲ではありませんでした）。"
        )

    if not layers:
        raise RuntimeError(
            f"'{filename}' から有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
        )

    combined = gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs="EPSG:4326")
    print(f"有効なハザードポリゴンを {len(combined)}件読み込みました。")
    combined_hazard_types = sorted(combined["hazard_type"].unique())
    if len(combined_hazard_types) == 1:
        print(f"hazard_type='{combined_hazard_types[0]}'")
    else:
        print(f"hazard_type: {', '.join(combined_hazard_types)}")
    return combined


hazard_gdf = None

if ENABLE_HAZARD_CHECK:
    print("ハザード区域のデータ（.geojson または国・県等の公式配布ZIP）をアップロードしてください（複数選択可）。")
    uploaded_hazards = files.upload()

    _stage_start = time.perf_counter()

    hazard_layers = []
    hazard_load_errors = []
    for hazard_filename, hazard_bytes in uploaded_hazards.items():
        detected_hazard_type = detect_hazard_type(hazard_filename)
        if detected_hazard_type is not None:
            hazard_type = detected_hazard_type
            print(f"'{hazard_filename}'")
            print(f"→ ハザード種別を自動判定しました: {hazard_type}")
        else:
            hazard_type = input(
                f"'{hazard_filename}' のハザード種別を自動判定できませんでした。\n"
                "ハザード種別名を入力してください（例: 洪水, 土砂災害, 津波, 高潮。"
                "ZIP内にサブフォルダがあれば、フォルダ名がカテゴリとして自動的に追加されます）: "
            ).strip()
            if not hazard_type:
                hazard_type = hazard_filename
        # 選択した複数ファイルのうち一部だけ読込失敗した状態のまま候補・ハザード判定へ進めると、
        # 判定結果のFalse（有効な座標で判定した結果、区域外）が「選択した全ハザードデータで
        # 判定済み」と誤認されかねない。そのため1ファイルの読込失敗（形式非対応・ZIP内に
        # geojson/shpが無い・有効なポリゴンが無い等）で即座に生のtracebackで止めはしないが、
        # ここでは握りつぶさず内容を集約し、全ファイルの検査後に1件でも失敗があれば処理を
        # 停止する（5種類全部を必須にはしないが、選択したファイルは全て正常に読めることを
        # 結果生成の条件にする。fail-closed）。
        try:
            layer = load_hazard_upload(hazard_filename, hazard_bytes, hazard_type)
        except (RuntimeError, ValueError) as error:
            hazard_load_errors.append((hazard_filename, str(error)))
            continue
        hazard_layers.append(layer)

    if hazard_load_errors:
        error_list = "\n".join(f"  - {name}: {reason}" for name, reason in hazard_load_errors)
        raise RuntimeError(
            "ハザードデータの一部を読み込めなかったため、判定結果が不完全になることを防ぐため"
            "処理を停止しました。\n"
            "\n読み込めなかったファイル:\n"
            f"{error_list}\n"
            "\nファイルを確認するか、今回判定に使用しないファイルは選択から外して、"
            "「ハザードデータ読込」セルを再実行してください。"
        )

    if hazard_layers:
        hazard_gdf = gpd.GeoDataFrame(pd.concat(hazard_layers, ignore_index=True), crs="EPSG:4326")

    if hazard_gdf is None or len(hazard_gdf) == 0:
        raise RuntimeError(
            "ENABLE_HAZARD_CHECK=True ですが、有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
            "GeoJSON/ZIPファイルが正しくアップロードされているか、ジオメトリ形式を確認してください。"
            "ハザード判定を行わない場合は、上の「利用者設定」で ENABLE_HAZARD_CHECK=False にしてください。"
        )

    print(f"ハザードデータを合計 {len(hazard_gdf)}件読み込みました。")

    TIMINGS["hazard"] = time.perf_counter() - _stage_start
    print(f"[処理時間] ハザードデータ読込・統合: {TIMINGS['hazard']:.2f}秒")
else:
    print("ENABLE_HAZARD_CHECK=False のため、ハザードデータの読込をスキップします。")
    TIMINGS["hazard"] = 0.0


In [ ]:
# ===== 距離計算・ハザード判定関数 =====
# 各要支援者について、有効な避難所すべてとの直線距離（geopy.distance.geodesic、メートル単位）を計算し、
# 近い順に TOP_N 件を候補として算出します（道路距離ではありません）。並び替えは丸める前の距離で行い、
# メートル単位への丸め（小数1桁）はCSVに出力する値を作成する際にのみ行います。距離が同一の場合でも
# 結果順が実行ごとにばらつかないよう、避難所名を用いて順序を安定させます。
#
# ハザード判定関数は、要支援者地点・候補避難所地点がハザード区域の内部または境界上にあるか、また
# 両地点を結ぶ直線がハザード区域と交差するかを判定します（直線交差は道路上の避難経路判定ではありません）。
# 座標が欠損・範囲外で判定できない場合は、区域外(False)と混同しないよう NaN を返します。
# ハザード区域は10万件規模になるため、1回の判定ごとに全ポリゴンを調べると要支援者数に比例して
# 待ち時間が延びます。GeoPandasの空間インデックス（hazard_area.sindex。1度作れば以降は再利用
# されます）で交差し得るポリゴンだけに絞ってから判定します（判定結果は絞り込みの有無で変わりません）。


def build_shelter_records(shelters_df):
    """避難所DataFrameから、距離計算に使う項目だけを取り出したタプルのリストを作る。
    要支援者1人ごとにDataFrameを1行ずつ取り出し直すと人数分だけ無駄に時間がかかるため、
    最初に1回だけ作って全員で使い回す。名前は並び順を安定させるため文字列に揃える。"""
    has_disaster_info = "_disaster_support" in shelters_df.columns
    return [
        (
            str(shelter["name"]),
            shelter["latitude"],
            shelter["longitude"],
            shelter["_disaster_support"] if has_disaster_info else np.nan,
        )
        for _, shelter in shelters_df.iterrows()
    ]


def compute_candidates(resident_lat, resident_lon, shelter_records, top_n):
    """要支援者の座標から近い順に避難所候補を [(名前, 距離m, 緯度, 経度, 災害種別対応区分), ...] で返す。
    座標が欠損、または緯度・経度が有効範囲外であれば空リストを返す（geodesic()に不正値を渡さない）。
    距離は丸めずに返す（並び替え後、出力時にのみ丸める）。距離が同一の場合は避難所名で順序を
    安定させる。対応区分は災害種別列が無ければNaNのまま。"""
    if not is_valid_coordinate(resident_lat, resident_lon):
        return []

    resident_coord = (resident_lat, resident_lon)
    candidates = [
        (name, geodesic(resident_coord, (lat, lon)).meters, lat, lon, disaster_support)
        for name, lat, lon, disaster_support in shelter_records
    ]
    candidates.sort(key=lambda candidate: (candidate[1], candidate[0]))
    return candidates[:top_n]


def hazard_types_intersecting(geometry, hazard_area):
    """ジオメトリ（点または直線）と交差するハザード区域を調べ、
    (該当したか, 該当したhazard_typeを;で連結した文字列) を返す。"""
    hit_rows = hazard_area.sindex.query(geometry, predicate="intersects")
    hit_types = sorted(hazard_area["hazard_type"].iloc[hit_rows].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


def hazard_types_at_point(lat, lon, hazard_area):
    """座標がハザード区域の内部または境界上にあるかどうかと、該当するhazard_type（;区切り）を返す。
    座標が欠損・範囲外の場合は判定不能としてnp.nanを返す（区域外=Falseと混同しない）。"""
    if not is_valid_coordinate(lat, lon):
        return np.nan, np.nan
    return hazard_types_intersecting(Point(lon, lat), hazard_area)


def hazard_types_on_line(lat1, lon1, lat2, lon2, hazard_area):
    """2地点を結ぶ直線がハザード区域と交差するかどうかと、該当するhazard_typeを返す（道路経路上の判定ではない）。
    いずれかの座標が欠損・範囲外の場合は判定不能としてnp.nanを返す。"""
    if not is_valid_coordinate(lat1, lon1) or not is_valid_coordinate(lat2, lon2):
        return np.nan, np.nan
    return hazard_types_intersecting(LineString([(lon1, lat1), (lon2, lat2)]), hazard_area)


print("距離計算・ハザード判定関数を定義しました。")


In [ ]:
# ===== 候補算出とハザード判定の実行 =====
# 要支援者ごとに、避難所候補・距離・（ENABLE_HAZARD_CHECK=True の場合のみ）ハザード判定結果を組み立てます。
# 距離による候補順位はハザード判定結果によって変更されません。座標がない・不正な要支援者の行も削除せず、
# 候補・距離を空欄のまま保持します（match_status 列で ok / no_coordinates(座標欄が空欄) /
# invalid_coordinates(値は入っているが数値として読めない・範囲外) の状態を確認できます）。
# ハザードデータを読み込んでいない状態でこのセルを実行した場合は、全行が「区域外」と誤読される結果を
# 出さないよう、ここで処理を止めます。
# ハザード関連列は、有効な座標で実際に判定できた場合のみ True/False とし、要支援者座標が欠損・不正な場合や
# 候補自体が存在しない場合は NaN（判定不能）のまま出力します（候補避難所地点は常に有効座標を持つため、
# 候補が存在すれば通常どおり True/False で判定します）。
# 避難所一覧に災害種別列があった場合は candidate_N_disaster_support 列に、災害種別ごとの対応区分
# （例: 洪水:対応済み;高潮:2階以上であれば対応済み）を出力します（対応区分によって候補の順位変更・自動除外は行いません）。

# ハザード判定を行う設定なのにハザードデータが読み込まれていない場合は、ここで処理を止める。
# このまま進めると全行が False（＝有効な座標で判定した結果、区域外）として出力され、
# 「判定していない」ことと「ハザード区域ではない」ことの区別が付かなくなるため。
if ENABLE_HAZARD_CHECK and (hazard_gdf is None or len(hazard_gdf) == 0):
    raise RuntimeError(
        "ENABLE_HAZARD_CHECK=True ですが、ハザードデータが読み込まれていません。"
        "先に「ハザードデータ読込」セルを実行してください"
        "（ハザード判定を行わない場合は、上の「利用者設定」で ENABLE_HAZARD_CHECK=False にしてください）。"
    )

_stage_start = time.perf_counter()

# 避難所の一覧は全員分の距離計算で共通のため、1回だけ作って使い回す。
shelter_records = build_shelter_records(shelters_valid)

# 候補避難所は要支援者ごとに同じ避難所がくり返し選ばれるため、避難所地点のハザード判定は
# 避難所ごとに1回だけ行い、結果を使い回す（同じ判定を人数分くり返さない）。
shelter_hazard_results = {}


def shelter_hazard_at(shelter_lat, shelter_lon):
    key = (shelter_lat, shelter_lon)
    if key not in shelter_hazard_results:
        shelter_hazard_results[key] = hazard_types_at_point(shelter_lat, shelter_lon, hazard_gdf)
    return shelter_hazard_results[key]


result_records = []

# レビュー用HTMLの表示データ。CSVと候補順位がずれないよう、候補はこのループで得たものを
# そのまま使う（HTML用に別の候補順位計算を持たない）。避難所の緯度・経度はCSVへは追加せず、
# 地図表示のためにここだけで保持する。
review_rows = []

for lat, lon, coord_status in zip(resident_lat, resident_lon, resident_coord_status):
    candidates = compute_candidates(lat, lon, shelter_records, TOP_N)
    review_candidates = []

    # match_statusは、職員が1行を横方向に追ったときに最初に目に入るよう先頭へ置く
    # （候補・距離を見てから「実は座標が無効だった」と分かる並びにしない）。
    record = {"match_status": coord_status}

    if ENABLE_HAZARD_CHECK:
        record["resident_in_hazard"], record["resident_hazard_types"] = hazard_types_at_point(
            lat, lon, hazard_gdf
        )

    for n in range(1, TOP_N + 1):
        # 候補が無い（座標が無効、または避難所件数がTOP_Nに満たない）場合は、判定できなかった
        # ことを示すため全項目を空欄（NaN）のままにする。
        shelter_name = distance_m = disaster_support = np.nan
        shelter_in_hazard = shelter_hazard_types = np.nan
        line_intersects = line_hazard_types = np.nan

        if n <= len(candidates):
            shelter_name, distance_raw, shelter_lat, shelter_lon, disaster_support = candidates[n - 1]
            distance_m = round(distance_raw, 1)
            if ENABLE_HAZARD_CHECK:
                shelter_in_hazard, shelter_hazard_types = shelter_hazard_at(shelter_lat, shelter_lon)
                line_intersects, line_hazard_types = hazard_types_on_line(
                    lat, lon, shelter_lat, shelter_lon, hazard_gdf
                )
            review_candidates.append({
                "rank": n,
                "name": shelter_name,
                "latitude": shelter_lat,
                "longitude": shelter_lon,
                "distance_m": distance_m,
                "disaster_support": disaster_support if HAS_DISASTER_TYPE_COLUMNS else None,
                "shelter_in_hazard": shelter_in_hazard,
                "shelter_hazard_types": shelter_hazard_types,
                "straight_line_intersects_hazard": line_intersects,
                "straight_line_hazard_types": line_hazard_types,
            })

        record[f"candidate_{n}"] = shelter_name
        record[f"distance_{n}_m"] = distance_m
        if HAS_DISASTER_TYPE_COLUMNS:
            record[f"candidate_{n}_disaster_support"] = disaster_support
        if ENABLE_HAZARD_CHECK:
            record[f"candidate_{n}_shelter_in_hazard"] = shelter_in_hazard
            record[f"candidate_{n}_shelter_hazard_types"] = shelter_hazard_types
            record[f"candidate_{n}_straight_line_intersects_hazard"] = line_intersects
            record[f"candidate_{n}_straight_line_hazard_types"] = line_hazard_types

    result_records.append(record)
    review_rows.append({
        "latitude": lat if coord_status == "ok" else None,
        "longitude": lon if coord_status == "ok" else None,
        "candidates": review_candidates,
    })

results_df = pd.DataFrame(result_records)

# 前回の結果CSV（assigned_shelters.csv）をそのまま再投入された場合、入力側にも同じ名前の列が
# 残っている。そのまま連結すると同名の列が二重になり、以降の集計がエラーで落ちるため、
# このNotebookが作る列は入力側から取り除いてから連結する。
#
# 取り除く対象は、下の一覧に挙げた「このNotebookが作る列名そのもの」に限る。
# candidate_1_備考 のように職員が独自に追加した列は、名前が似ていても取り除かない
# （入力されたデータを黙って消さないため）。
GENERATED_COLUMNS = ("match_status", "resident_in_hazard", "resident_hazard_types")

# 候補ごとに作る列。候補番号(N)は、前回と異なる TOP_N で再実行される場合があるため、
# 今回のTOP_Nに限定せず数字として判定する。
GENERATED_CANDIDATE_COLUMNS = (
    "candidate_{n}",
    "distance_{n}_m",
    "candidate_{n}_disaster_support",
    "candidate_{n}_shelter_in_hazard",
    "candidate_{n}_shelter_hazard_types",
    "candidate_{n}_straight_line_intersects_hazard",
    "candidate_{n}_straight_line_hazard_types",
)


def is_generated_column(column_name):
    """このNotebookが結果として作る列名かどうかを返す（上の一覧と完全に一致する名前のみTrue）。"""
    if column_name in GENERATED_COLUMNS:
        return True
    return any(
        re.fullmatch(template.format(n=r"\d+"), column_name)
        for template in GENERATED_CANDIDATE_COLUMNS
    )


reused_columns = [c for c in residents.columns if is_generated_column(c)]
if reused_columns:
    print(f"入力CSVに前回の結果列が含まれていたため、今回の結果で置き換えます: {', '.join(reused_columns)}")

final_df = pd.concat(
    [residents.drop(columns=reused_columns).reset_index(drop=True), results_df], axis=1
)

TIMINGS["candidates"] = time.perf_counter() - _stage_start
print("候補算出が完了しました。")
print(f"[処理時間] 避難所候補算出＋ハザード判定: {TIMINGS['candidates']:.2f}秒")


In [ ]:
# ===== レビュー用HTML生成（補助成果物） =====
# assigned_shelters.csv が正式なデータ成果物で、ここで作るHTMLはその結果を職員が地図上で
# 目視確認するための補助成果物です。避難所の割り当て結果ではありません。
#
# 出力は sheltermatch_review.zip 1つだけです（要支援者ごとのHTMLは作りません）。
#   sheltermatch_review/review.html          … 全要支援者のデータをJSONとして埋め込んだ1ファイル
#   sheltermatch_review/assets/hazard_*.png  … ハザード区域の表示用画像（大分類ごとに1枚）
#   sheltermatch_review/assets/js, css, images … Leaflet本体（地図ライブラリ）
#
# 地図ライブラリ（Leaflet）はZIPへ同梱するため、展開したフォルダだけで、外部通信なしに
# review.html を開けます。背景地図（地理院タイル）だけは外部通信が必要なので既定では
# 読み込まず、画面のチェックボックスで必要なときだけONにできます。背景地図が無くても、
# 要支援者・候補避難所・直線・ハザード区域は確認できます。
#
# 約18万件のハザードポリゴンはブラウザへ渡さず、表示用の透過PNGへ描き出します。画像は
# Web地図と位置がずれないよう、表示範囲をEPSG:3857（Webメルカトル）へ変換した矩形の中で描きます。
# 表示用の画像を作るだけで、ハザード判定や修復はここでは一切行いません（本体で処理済みの
# hazard_gdf をそのまま使います）。

# 同梱する地図ライブラリ。バージョンを固定し、取得したファイルの内容をSHA-256で確認してから
# 同梱する（レビュー成果物だけで完結させるため）。
LEAFLET_VERSION = "1.9.4"
# unpkgはnpmパッケージの内容をそのまま配信する。dist/配下がJS・CSS・画像、
# パッケージ直下にライセンス本文があるため、配布元のパスをそれぞれ指定する。
LEAFLET_BASE_URL = f"https://unpkg.com/leaflet@{LEAFLET_VERSION}"
LEAFLET_FILES = [
    # (配布元のパス, 成果物内のパス, SHA-256)
    ("dist/leaflet.js", "js/leaflet.js",
     "db49d009c841f5ca34a888c96511ae936fd9f5533e90d8b2c4d57596f4e5641a"),
    ("dist/leaflet.css", "css/leaflet.css",
     "a7837102824184820dfa198d1ebcd109ff6d0ff9a2672a074b9a1b4d147d04c6"),
    ("dist/images/layers.png", "images/layers.png",
     "1dbbe9d028e292f36fcba8f8b3a28d5e8932754fc2215b9ac69e4cdecf5107c6"),
    ("dist/images/layers-2x.png", "images/layers-2x.png",
     "066daca850d8ffbef007af00b06eac0015728dee279c51f3cb6c716df7c42edf"),
    ("dist/images/marker-icon.png", "images/marker-icon.png",
     "574c3a5cca85f4114085b6841596d62f00d7c892c7b03f28cbfa301deb1dc437"),
    ("dist/images/marker-icon-2x.png", "images/marker-icon-2x.png",
     "00179c4c1ee830d3a108412ae0d294f55776cfeb085c60129a39aa6fc4ae2528"),
    ("dist/images/marker-shadow.png", "images/marker-shadow.png",
     "264f5c640339f042dd729062cfc04c17f8ea0f29882b538e3848ed8f10edb4da"),
    # Leaflet 1.9.4はBSD 2-Clause Licenseで配布されており、再配布時は著作権表示・条件・
    # 免責条項の保持が必須のため、公式LICENSE本文をそのまま同梱する。
    ("LICENSE", "leaflet-LICENSE.txt",
     "53e8dc25862014e4324741ca18fbe3611e11d42ef69f59f86ea8c5389647d4cb"),
]

REVIEW_PACKAGE_NAME = "sheltermatch_review"
REVIEW_ZIP_FILENAME = f"{REVIEW_PACKAGE_NAME}.zip"
REVIEW_IMAGE_WIDTH = 1600  # 表示用PNGの横幅（px）

# 表示用のハザード大分類。解析結果として保持している詳細なhazard_typeは変更しない。
HAZARD_DISPLAY_GROUPS = [
    ("tsunami", "津波", "津波", "#0891b2"),
    ("flood", "洪水", "洪水", "#2563eb"),
    ("storm_surge", "高潮", "高潮", "#7c3aed"),
    ("landslide", "土砂災害", "土砂災害", "#b45309"),
    ("other", "その他", None, "#6b7280"),
]


def bundle_leaflet(assets_dir):
    """地図ライブラリ（Leaflet）を取得して成果物へ同梱する。

    取得できるのはNotebookを実行している環境（Google Colab等）だけで構わない。同梱した後の
    review.html は、展開したフォルダだけで外部通信なしに開ける。
    取得したファイルは、内容がバージョン固定のものと同じかSHA-256で確認してから書き込む。

    CSSは画像を 'images/...' という相対パスで参照するため、CSSを css/ へ置くとパスがずれる。
    利用者が中身を追いやすい assets/images/ へ画像をまとめたうえで、CSS側の参照を
    '../images/...' へ書き換える。"""
    for source_path, package_path, expected_digest in LEAFLET_FILES:
        url = f"{LEAFLET_BASE_URL}/{source_path}"
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
        except Exception as error:
            raise RuntimeError(
                f"地図ライブラリ（Leaflet {LEAFLET_VERSION}）を取得できませんでした: {url}\n"
                f"（詳細: {error}）\n"
                "結果CSVは出力済みです。レビュー用HTMLだけを作り直す場合は、"
                "インターネットに接続できる状態でこのセルを再実行してください。"
            ) from error

        content = response.content
        actual_digest = hashlib.sha256(content).hexdigest()
        if actual_digest != expected_digest:
            raise RuntimeError(
                f"地図ライブラリの内容が想定と異なります: {url}\n"
                f"  期待 {expected_digest}\n  実際 {actual_digest}"
            )

        if package_path.endswith(".css"):
            content = content.decode("utf-8").replace("url(images/", "url(../images/").encode("utf-8")

        target = assets_dir / package_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)

    print(f"地図ライブラリ（Leaflet {LEAFLET_VERSION}）を成果物へ同梱しました。")


def hazard_display_group(hazard_type):
    """詳細なhazard_typeを、表示用の大分類キーへ寄せる（元のhazard_typeは変更しない）。"""
    text = str(hazard_type)
    for key, _label, prefix, _color in HAZARD_DISPLAY_GROUPS:
        if prefix is not None and text.startswith(prefix):
            return key
    return "other"


def split_hazard_types(value):
    """';'区切りのhazard_typeを一覧へ分解する（判定していない・該当なしは空の一覧）。"""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return []
    return [part for part in str(value).split(";") if part]


def review_display_bounds(latitudes, longitudes):
    """有効な要支援者地点・避難所地点の全体範囲に余白を付けた共通表示範囲を返す
    （min_lat, min_lon, max_lat, max_lon）。すべてのハザードPNGでこの範囲を使う。"""
    min_lat, max_lat = float(np.min(latitudes)), float(np.max(latitudes))
    min_lon, max_lon = float(np.min(longitudes)), float(np.max(longitudes))
    lat_margin = max((max_lat - min_lat) * 0.08, 0.005)
    lon_margin = max((max_lon - min_lon) * 0.08, 0.005)
    return (min_lat - lat_margin, min_lon - lon_margin, max_lat + lat_margin, max_lon + lon_margin)


def render_hazard_images(hazard_area, display_bounds, assets_dir):
    """ハザード区域を大分類ごとの透過PNGへ描き出し、HTMLへ渡すレイヤー情報を返す。

    位置合わせのため、表示範囲をEPSG:3857へ変換した矩形の中で描く（緯度経度のまま描いた画像を
    引き伸ばすと、Web地図上でずれるため）。同じ種別のポリゴンが重なった場所だけ濃くならないよう、
    PNGは不透明な単色で描き、半透明表示はHTML側のレイヤー不透明度で行う。"""
    min_lat, min_lon, max_lat, max_lon = display_bounds
    display_box = box(min_lon, min_lat, max_lon, max_lat)

    # 表示範囲と交差するポリゴンだけに絞る（画像に写らないポリゴンは描かない）
    visible_rows = hazard_area.sindex.query(display_box, predicate="intersects")
    visible = hazard_area.iloc[visible_rows]
    if len(visible) == 0:
        return []

    mercator_bounds = gpd.GeoSeries([display_box], crs="EPSG:4326").to_crs(epsg=3857).total_bounds
    minx, miny, maxx, maxy = mercator_bounds
    image_height = max(1, int(round(REVIEW_IMAGE_WIDTH * (maxy - miny) / (maxx - minx))))

    visible = visible.to_crs(epsg=3857)
    display_groups = visible["hazard_type"].map(hazard_display_group)

    layers = []
    for key, label, _prefix, color in HAZARD_DISPLAY_GROUPS:
        group = visible[display_groups == key]
        if len(group) == 0:
            continue

        figure = plt.figure(figsize=(REVIEW_IMAGE_WIDTH / 100, image_height / 100), dpi=100)
        axes = figure.add_axes([0, 0, 1, 1])
        axes.set_xlim(minx, maxx)
        axes.set_ylim(miny, maxy)
        axes.set_axis_off()
        group.plot(ax=axes, color=color, linewidth=0, antialiased=False)
        image_name = f"hazard_{key}.png"
        figure.savefig(assets_dir / image_name, dpi=100, transparent=True)
        plt.close(figure)

        layers.append({
            "key": key,
            "label": label,
            "color": color,
            "image": f"assets/{image_name}",
            # ImageOverlayへ渡す緯度経度の矩形。上でEPSG:3857へ変換したのと同じ矩形。
            "bounds": [[min_lat, min_lon], [max_lat, max_lon]],
            "polygon_count": int(len(group)),
        })
    return layers


def json_value(value):
    """NaN等をそのままJSONにできないため、Pythonの標準的な値へ整える。"""
    if value is None:
        return None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, str):
        return value
    if pd.isna(value):
        return None
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    return str(value)


def build_review_data(final_df, review_rows, hazard_layers, top_n):
    """結果CSVと同じ実行結果から、レビューHTMLへ埋め込むデータを組み立てる。
    候補の順位・距離は候補算出ループで得たものをそのまま使い、ここで計算し直さない
    （CSVとHTMLで候補順位がずれないようにするため）。"""
    residents = []
    for position in range(len(final_df)):
        row = final_df.iloc[position]
        review_row = review_rows[position]

        candidates = []
        hazard_groups = set()
        for candidate in review_row["candidates"]:
            shelter_types = split_hazard_types(candidate["shelter_hazard_types"])
            line_types = split_hazard_types(candidate["straight_line_hazard_types"])
            hazard_groups.update(hazard_display_group(t) for t in shelter_types + line_types)
            candidates.append({
                "rank": candidate["rank"],
                "name": json_value(candidate["name"]),
                "latitude": json_value(candidate["latitude"]),
                "longitude": json_value(candidate["longitude"]),
                "distance_m": json_value(candidate["distance_m"]),
                "disaster_support": json_value(candidate["disaster_support"]),
                "shelter_in_hazard": json_value(candidate["shelter_in_hazard"]),
                "shelter_hazard_types": shelter_types,
                "straight_line_intersects_hazard": json_value(
                    candidate["straight_line_intersects_hazard"]
                ),
                "straight_line_hazard_types": line_types,
            })

        resident_types = split_hazard_types(row.get("resident_hazard_types"))
        hazard_groups.update(hazard_display_group(t) for t in resident_types)

        latitude = json_value(review_row["latitude"])
        longitude = json_value(review_row["longitude"])
        residents.append({
            "resident_id": json_value(row["resident_id"]),
            "address": json_value(row.get("address")),
            "latitude": latitude,
            "longitude": longitude,
            "has_coordinates": latitude is not None and longitude is not None,
            "match_status": json_value(row["match_status"]),
            "resident_in_hazard": json_value(row.get("resident_in_hazard")),
            "resident_hazard_types": resident_types,
            "hazard_groups": sorted(hazard_groups),
            "candidates": candidates,
        })

    return {
        "generated_at": time.strftime("%Y-%m-%d %H:%M"),
        "top_n": top_n,
        "hazard_layers": hazard_layers,
        "residents": residents,
    }


def verify_review_data(review_data, final_df, top_n):
    """HTMLへ埋め込むデータが結果CSVと一致していることを確認する。
    1件でも食い違えばHTMLを作らずに処理を止める（CSVと見比べて判断できない資料を出さないため）。"""
    problems = []
    residents = review_data["residents"]

    if len(residents) != len(final_df):
        problems.append(f"要支援者件数が一致しません（HTML {len(residents)}件 / CSV {len(final_df)}件）")
    else:
        for position, resident in enumerate(residents):
            row = final_df.iloc[position]
            label = f"CSV{position + 2}行目"

            if str(resident["resident_id"]) != str(row["resident_id"]):
                problems.append(
                    f"{label}: resident_idが一致しません"
                    f"（HTML '{resident['resident_id']}' / CSV '{row['resident_id']}'）"
                )

            by_rank = {candidate["rank"]: candidate for candidate in resident["candidates"]}
            for rank in range(1, top_n + 1):
                candidate = by_rank.get(rank)
                csv_name = row.get(f"candidate_{rank}")

                if pd.isna(csv_name):
                    if candidate is not None:
                        problems.append(f"{label}: CSVに候補{rank}が無いのにHTMLにはあります")
                    continue
                if candidate is None:
                    problems.append(f"{label}: CSVの候補{rank}がHTMLにありません")
                    continue

                for field, csv_column in (
                    ("name", f"candidate_{rank}"),
                    ("distance_m", f"distance_{rank}_m"),
                    ("shelter_in_hazard", f"candidate_{rank}_shelter_in_hazard"),
                    ("straight_line_intersects_hazard",
                     f"candidate_{rank}_straight_line_intersects_hazard"),
                ):
                    if csv_column not in final_df.columns:
                        continue
                    if candidate[field] != json_value(row[csv_column]):
                        problems.append(
                            f"{label}: 候補{rank}の{csv_column}が一致しません"
                            f"（HTML {candidate[field]!r} / CSV {json_value(row[csv_column])!r}）"
                        )

                for field, csv_column in (
                    ("shelter_hazard_types", f"candidate_{rank}_shelter_hazard_types"),
                    ("straight_line_hazard_types", f"candidate_{rank}_straight_line_hazard_types"),
                ):
                    if csv_column not in final_df.columns:
                        continue
                    if candidate[field] != split_hazard_types(row[csv_column]):
                        problems.append(f"{label}: 候補{rank}の{csv_column}が一致しません")

            if "resident_in_hazard" in final_df.columns:
                if resident["resident_in_hazard"] != json_value(row["resident_in_hazard"]):
                    problems.append(f"{label}: resident_in_hazardが一致しません")
                if resident["resident_hazard_types"] != split_hazard_types(row["resident_hazard_types"]):
                    problems.append(f"{label}: resident_hazard_typesが一致しません")

    if problems:
        raise RuntimeError(
            "レビューHTMLの内容が結果CSVと一致しないため、HTMLを作らずに処理を停止しました。\n"
            + "\n".join(f"  - {problem}" for problem in problems[:20])
            + (f"\n  ほか{len(problems) - 20}件" if len(problems) > 20 else "")
        )


def embed_review_json(review_data):
    """HTMLへ安全に埋め込める形のJSON文字列を返す。住所等の利用者データがそのまま
    HTMLとして解釈されないよう、'<' '>' '&' をJSONのエスケープ表現へ置き換える
    （'</script>' でscriptタグを壊せないようにするため）。"""
    text = json.dumps(review_data, ensure_ascii=False, allow_nan=False)
    for character, escaped in (
        ("<", "\\u003c"), (">", "\\u003e"), ("&", "\\u0026"),
        # 行区切り扱いの文字はJavaScriptの文字列を壊すため、これも置き換える
        (chr(0x2028), "\\u2028"), (chr(0x2029), "\\u2029"),
    ):
        text = text.replace(character, escaped)
    return text


def build_review_package(final_df, review_rows, hazard_area, shelters_df, top_n, output_dir):
    """レビュー用のHTML・PNGを作り、ZIPへまとめてそのパスを返す。"""
    package_dir = Path(output_dir) / REVIEW_PACKAGE_NAME
    assets_dir = package_dir / "assets"
    if package_dir.exists():
        shutil.rmtree(package_dir)
    assets_dir.mkdir(parents=True)

    latitudes = [row["latitude"] for row in review_rows if row["latitude"] is not None]
    longitudes = [row["longitude"] for row in review_rows if row["longitude"] is not None]
    latitudes += list(shelters_df["latitude"])
    longitudes += list(shelters_df["longitude"])

    bundle_leaflet(assets_dir)

    hazard_layers = []
    if hazard_area is not None and len(hazard_area) > 0 and latitudes:
        display_bounds = review_display_bounds(latitudes, longitudes)
        hazard_layers = render_hazard_images(hazard_area, display_bounds, assets_dir)

    review_data = build_review_data(final_df, review_rows, hazard_layers, top_n)
    verify_review_data(review_data, final_df, top_n)

    html = REVIEW_HTML_TEMPLATE.replace("__REVIEW_DATA_JSON__", embed_review_json(review_data))
    (package_dir / "review.html").write_text(html, encoding="utf-8")

    zip_path = Path(output_dir) / REVIEW_ZIP_FILENAME
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for path in sorted(package_dir.rglob("*")):
            if path.is_file():
                zip_file.write(path, path.relative_to(output_dir))

    print(f"レビュー用HTMLを作成しました（要支援者 {len(review_data['residents'])}件）。")
    print("ZIPを展開して review.html を開けば、外部通信なしで利用できます"
          "（背景地図だけは画面のチェックボックスでONにでき、外部通信が必要です）。")
    if hazard_layers:
        print("ハザード表示用PNG:")
        for layer in hazard_layers:
            print(f"  {layer['label']}: {layer['polygon_count']}ポリゴン → {layer['image']}")
    else:
        print("ハザード判定を行っていないため、ハザード表示用PNGは作成していません。")
    return zip_path


REVIEW_HTML_TEMPLATE = "<!DOCTYPE html>\n<html lang=\"ja\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>避難所候補レビュー（sheltermatch）</title>\n<link rel=\"stylesheet\" href=\"assets/css/leaflet.css\">\n<style>\n  :root {\n    --border: #d7dbe0;\n    --muted: #5b6672;\n    --bg: #f5f6f8;\n    --panel: #ffffff;\n    --accent: #1b4d89;\n    --warn-bg: #fff4e5;\n    --warn-border: #e0a561;\n  }\n  * { box-sizing: border-box; }\n  html, body { height: 100%; margin: 0; }\n  body {\n    font-family: \"Hiragino Kaku Gothic ProN\", \"Yu Gothic\", Meiryo, sans-serif;\n    color: #1d2329; background: var(--bg); font-size: 14px;\n  }\n  #app { display: flex; flex-direction: column; height: 100vh; }\n  header {\n    padding: 8px 14px; background: var(--panel); border-bottom: 1px solid var(--border);\n  }\n  header h1 { font-size: 16px; margin: 0 0 4px; }\n  .notice { font-size: 12px; color: var(--muted); line-height: 1.5; }\n  .notice strong { color: #8a4b08; }\n  #main { display: flex; flex: 1; min-height: 0; }\n  #sidebar {\n    width: 320px; flex: none; background: var(--panel); border-right: 1px solid var(--border);\n    display: flex; flex-direction: column; min-height: 0;\n  }\n  #sidebar .section { padding: 10px 12px; border-bottom: 1px solid var(--border); }\n  #sidebar .section.grow { flex: 1; min-height: 0; overflow-y: auto; }\n  h2 { font-size: 13px; margin: 0 0 6px; color: var(--muted); font-weight: 600; }\n  input[type=\"search\"] { width: 100%; padding: 6px 8px; border: 1px solid var(--border); border-radius: 4px; font-size: 14px; }\n  .count { font-size: 12px; color: var(--muted); margin-top: 6px; }\n  ul.results { list-style: none; margin: 0; padding: 0; }\n  ul.results li {\n    padding: 6px 8px; border-radius: 4px; cursor: pointer; border: 1px solid transparent;\n    line-height: 1.4;\n  }\n  ul.results li:hover { background: #eef2f7; }\n  ul.results li.selected { background: #e3edf9; border-color: var(--accent); }\n  .rid { font-weight: 600; }\n  .addr { font-size: 12px; color: var(--muted); word-break: break-all; }\n  .badge {\n    display: inline-block; font-size: 11px; padding: 0 5px; border-radius: 8px;\n    border: 1px solid var(--warn-border); background: var(--warn-bg); color: #8a4b08; margin-left: 4px;\n  }\n  .nav { display: flex; gap: 6px; align-items: center; }\n  button {\n    padding: 5px 10px; border: 1px solid var(--border); background: #fff; border-radius: 4px;\n    cursor: pointer; font-size: 13px;\n  }\n  button:hover:enabled { background: #eef2f7; }\n  button:disabled { opacity: .45; cursor: default; }\n  #content { flex: 1; display: flex; flex-direction: column; min-width: 0; min-height: 0; }\n  #map { flex: 1; min-height: 240px; background: #f0f3f6; }\n  .map-note { font-size: 12px; color: var(--muted); margin-top: 6px; line-height: 1.5; }\n  .map-note.warn { color: #8a4b08; }\n  #map-fallback { padding: 16px; color: var(--muted); display: none; }\n  #detail { border-top: 1px solid var(--border); background: var(--panel); padding: 10px 12px; max-height: 42%; overflow-y: auto; }\n  #selected-summary { margin-bottom: 8px; line-height: 1.6; }\n  #selected-summary .title { font-size: 15px; font-weight: 600; }\n  table { border-collapse: collapse; width: 100%; font-size: 13px; }\n  th, td { border: 1px solid var(--border); padding: 5px 7px; text-align: left; vertical-align: top; }\n  th { background: #eef1f5; font-weight: 600; white-space: nowrap; }\n  tbody tr { cursor: pointer; }\n  tbody tr:hover { background: #f2f6fb; }\n  tbody tr.active { background: #e3edf9; }\n  .rank-chip {\n    display: inline-block; width: 20px; height: 20px; line-height: 20px; text-align: center;\n    border-radius: 50%; color: #fff; font-weight: 700; font-size: 12px;\n  }\n  .layers { display: flex; flex-wrap: wrap; gap: 4px 12px; }\n  .layers label { display: flex; align-items: center; gap: 5px; font-size: 13px; }\n  .swatch { width: 12px; height: 12px; border-radius: 2px; display: inline-block; border: 1px solid rgba(0,0,0,.25); }\n  .empty { color: var(--muted); padding: 8px 0; }\n  .hazard-yes { color: #a13a0c; font-weight: 600; }\n  .hazard-no { color: var(--muted); }\n  .marker-pin {\n    width: 22px; height: 22px; border-radius: 50%; border: 2px solid #fff;\n    box-shadow: 0 0 0 1px rgba(0,0,0,.4); color: #fff; font-weight: 700; font-size: 12px;\n    line-height: 18px; text-align: center;\n  }\n</style>\n</head>\n<body>\n<div id=\"app\">\n  <header>\n    <h1>避難所候補レビュー（sheltermatch）</h1>\n    <div class=\"notice\">\n      避難所候補は<strong>距離による候補であり、最終的な割り当てではありません</strong>。\n      表示している線は<strong>直線であり、避難経路を示すものではありません</strong>（直線交差判定に使っている線です）。\n      最終的な避難先は職員が判断してください。\n      <span id=\"generated-at\"></span>\n    </div>\n  </header>\n  <div id=\"main\">\n    <div id=\"sidebar\">\n      <div class=\"section\">\n        <h2>要支援者を検索</h2>\n        <input type=\"search\" id=\"search\" placeholder=\"resident_id または 住所で絞り込み\" autocomplete=\"off\">\n        <div class=\"count\" id=\"count\"></div>\n      </div>\n      <div class=\"section grow\">\n        <ul class=\"results\" id=\"results\"></ul>\n        <div class=\"count\" id=\"more-note\"></div>\n      </div>\n      <div class=\"section\">\n        <div class=\"nav\">\n          <button id=\"prev\" type=\"button\">← 前へ</button>\n          <button id=\"next\" type=\"button\">次へ →</button>\n          <span class=\"count\" id=\"position\"></span>\n        </div>\n      </div>\n      <div class=\"section\">\n        <h2>地図の表示</h2>\n        <div class=\"layers\">\n          <label>\n            <input type=\"checkbox\" id=\"basemap-toggle\">\n            <span>背景地図を表示（インターネット接続が必要）</span>\n          </label>\n        </div>\n        <div class=\"map-note\" id=\"basemap-note\"></div>\n        <div class=\"map-note\" id=\"view-bounds\"></div>\n      </div>\n      <div class=\"section\" id=\"layer-section\">\n        <h2>ハザード区域の表示</h2>\n        <div class=\"layers\" id=\"layers\"></div>\n        <div class=\"nav\" style=\"margin-top:8px\">\n          <button id=\"layers-all\" type=\"button\">すべて表示</button>\n          <button id=\"layers-none\" type=\"button\">すべて非表示</button>\n        </div>\n        <div class=\"count\">要支援者を選ぶと、その人に関係する種別を自動で表示します。</div>\n      </div>\n    </div>\n    <div id=\"content\">\n      <div id=\"map\"></div>\n      <div id=\"map-fallback\"></div>\n      <div id=\"detail\">\n        <div id=\"selected-summary\"></div>\n        <div id=\"candidates\"></div>\n      </div>\n    </div>\n  </div>\n</div>\n\n<script type=\"application/json\" id=\"review-data\">__REVIEW_DATA_JSON__</script>\n<script src=\"assets/js/leaflet.js\"></script>\n<script>\n(function () {\n  \"use strict\";\n\n  var DATA = JSON.parse(document.getElementById(\"review-data\").textContent);\n  var RESIDENTS = DATA.residents || [];\n  var HAZARD_LAYERS = DATA.hazard_layers || [];\n  var RESULT_LIMIT = 50;\n  var RANK_COLORS = [\"#c62828\", \"#ef6c00\", \"#b58900\"];\n  var RESIDENT_COLOR = \"#14213d\";\n\n  var map = null;\n  var featureLayer = null;\n  var graticuleLayer = null;\n  var basemapLayer = null;\n  var overlays = {};\n  var enabledLayers = {};\n  var filtered = [];\n  var selectedIndex = -1;\n  var activeRank = null;\n\n  function el(id) { return document.getElementById(id); }\n\n  function setText(node, value) { node.textContent = value == null ? \"\" : String(value); }\n\n  function createRow(tag, className) {\n    var node = document.createElement(tag);\n    if (className) { node.className = className; }\n    return node;\n  }\n\n  // Leafletのツールチップは、文字列を渡すとHTMLとして解釈される。避難所名のような\n  // 利用者データがHTMLとして扱われないよう、必ずDOM要素にして渡す。\n  function textTooltip(text) {\n    var node = document.createElement(\"span\");\n    node.textContent = text;\n    return node;\n  }\n\n  // ---- 地図の初期化 ----\n  // Leaflet本体はこの成果物へ同梱しているため、外部通信なしでも地図は動く。\n  // 背景地図（地理院タイル）だけは外部通信が必要なので、既定では読み込まず、\n  // 利用者が必要なときだけONにできるようにしている。\n  function initMap() {\n    if (typeof L === \"undefined\") {\n      el(\"map\").style.display = \"none\";\n      var fallback = el(\"map-fallback\");\n      fallback.style.display = \"block\";\n      setText(fallback,\n        \"地図ライブラリ（Leaflet）を読み込めませんでした。review.html と同じ場所の \" +\n        \"assets フォルダごと展開されているか確認してください。\" +\n        \"検索・候補一覧はこのまま利用できます。\");\n      return;\n    }\n    map = L.map(\"map\").setView([26.125, 127.68], 13);\n    featureLayer = L.layerGroup().addTo(map);\n    graticuleLayer = L.layerGroup().addTo(map);\n    buildOverlays();\n    map.on(\"moveend zoomend\", drawGraticule);\n    drawGraticule();\n  }\n\n  // ---- 背景地図（任意・外部通信が必要） ----\n  function setBasemapEnabled(enabled) {\n    var note = el(\"basemap-note\");\n    if (!map) { return; }\n    if (!enabled) {\n      if (basemapLayer && map.hasLayer(basemapLayer)) { map.removeLayer(basemapLayer); }\n      setText(note, \"\");\n      return;\n    }\n    try {\n      if (!basemapLayer) {\n        basemapLayer = L.tileLayer(\"https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png\", {\n          maxZoom: 18,\n          attribution: \"地理院タイル（国土地理院）\"\n        });\n        // 通信できない環境ではタイルが取れないが、他の表示は動き続ける\n        basemapLayer.on(\"tileerror\", function () {\n          note.className = \"map-note warn\";\n          setText(note, \"背景地図を取得できませんでした（外部通信が必要です）。\"\n            + \"背景地図なしでも、要支援者・候補避難所・直線・ハザード区域は確認できます。\");\n        });\n      }\n      note.className = \"map-note\";\n      setText(note, \"背景地図を読み込んでいます…\");\n      basemapLayer.addTo(map);\n      basemapLayer.once(\"load\", function () {\n        if (note.className !== \"map-note warn\") { setText(note, \"\"); }\n      });\n    } catch (error) {\n      note.className = \"map-note warn\";\n      setText(note, \"背景地図を表示できませんでした。ほかの表示はそのまま利用できます。\");\n    }\n  }\n\n  // ---- 経緯度グリッド（背景地図が無いときの位置の手がかり） ----\n  var GRATICULE_STEPS = [1, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001];\n\n  function graticuleStep(span) {\n    for (var i = 0; i < GRATICULE_STEPS.length; i += 1) {\n      if (span / GRATICULE_STEPS[i] >= 4) { return GRATICULE_STEPS[i]; }\n    }\n    return GRATICULE_STEPS[GRATICULE_STEPS.length - 1];\n  }\n\n  function drawGraticule() {\n    if (!map || !graticuleLayer) { return; }\n    graticuleLayer.clearLayers();\n    var bounds = map.getBounds();\n    var south = bounds.getSouth(), north = bounds.getNorth();\n    var west = bounds.getWest(), east = bounds.getEast();\n    var step = graticuleStep(Math.max(north - south, east - west));\n    // 候補の直線と見分けられるよう、グリッド線には専用のクラス名を付ける\n    var style = {\n      className: \"graticule-line\",\n      color: \"#9aa5b1\", weight: 1, opacity: 0.5, interactive: false\n    };\n\n    for (var lat = Math.ceil(south / step) * step; lat <= north; lat += step) {\n      L.polyline([[lat, west], [lat, east]], style).addTo(graticuleLayer);\n    }\n    for (var lon = Math.ceil(west / step) * step; lon <= east; lon += step) {\n      L.polyline([[south, lon], [north, lon]], style).addTo(graticuleLayer);\n    }\n\n    setText(el(\"view-bounds\"),\n      \"表示範囲 緯度 \" + south.toFixed(4) + \"〜\" + north.toFixed(4)\n      + \" / 経度 \" + west.toFixed(4) + \"〜\" + east.toFixed(4)\n      + \"（グリッド間隔 \" + step + \"度）\");\n  }\n\n  function buildOverlays() {\n    HAZARD_LAYERS.forEach(function (layer) {\n      try {\n        overlays[layer.key] = L.imageOverlay(layer.image, layer.bounds, { opacity: 0.45 });\n      } catch (error) {\n        overlays[layer.key] = null;\n      }\n    });\n  }\n\n  // ---- ハザードレイヤ操作 ----\n  function renderLayerControls() {\n    var container = el(\"layers\");\n    container.textContent = \"\";\n    if (!HAZARD_LAYERS.length) {\n      el(\"layer-section\").style.display = \"none\";\n      return;\n    }\n    HAZARD_LAYERS.forEach(function (layer) {\n      var label = createRow(\"label\");\n      var input = document.createElement(\"input\");\n      input.type = \"checkbox\";\n      input.dataset.key = layer.key;\n      input.addEventListener(\"change\", function () {\n        setLayerEnabled(layer.key, input.checked);\n      });\n      var swatch = createRow(\"span\", \"swatch\");\n      swatch.style.background = layer.color;\n      var text = document.createElement(\"span\");\n      setText(text, layer.label);\n      label.appendChild(input);\n      label.appendChild(swatch);\n      label.appendChild(text);\n      container.appendChild(label);\n    });\n  }\n\n  function setLayerEnabled(key, enabled) {\n    enabledLayers[key] = enabled;\n    var checkbox = document.querySelector('#layers input[data-key=\"' + key + '\"]');\n    if (checkbox) { checkbox.checked = enabled; }\n    var overlay = overlays[key];\n    if (!map || !overlay) { return; }\n    if (enabled) {\n      if (!map.hasLayer(overlay)) { overlay.addTo(map); }\n    } else if (map.hasLayer(overlay)) {\n      map.removeLayer(overlay);\n    }\n  }\n\n  function setAllLayers(enabled) {\n    HAZARD_LAYERS.forEach(function (layer) { setLayerEnabled(layer.key, enabled); });\n  }\n\n  function applyRelevantLayers(resident) {\n    var relevant = {};\n    (resident.hazard_groups || []).forEach(function (key) { relevant[key] = true; });\n    HAZARD_LAYERS.forEach(function (layer) {\n      setLayerEnabled(layer.key, Boolean(relevant[layer.key]));\n    });\n  }\n\n  // ---- 検索・一覧 ----\n  function normalize(value) { return (value == null ? \"\" : String(value)).toLowerCase(); }\n\n  function applyFilter() {\n    var query = normalize(el(\"search\").value).trim();\n    filtered = [];\n    for (var i = 0; i < RESIDENTS.length; i += 1) {\n      var resident = RESIDENTS[i];\n      if (!query\n        || normalize(resident.resident_id).indexOf(query) >= 0\n        || normalize(resident.address).indexOf(query) >= 0) {\n        filtered.push(i);\n      }\n    }\n    renderResults();\n    if (filtered.indexOf(selectedIndex) < 0) {\n      select(filtered.length ? filtered[0] : -1);\n    } else {\n      updatePosition();\n    }\n  }\n\n  function renderResults() {\n    var list = el(\"results\");\n    list.textContent = \"\";\n    var shown = Math.min(filtered.length, RESULT_LIMIT);\n    for (var i = 0; i < shown; i += 1) {\n      list.appendChild(createResultItem(filtered[i]));\n    }\n    setText(el(\"count\"), \"該当 \" + filtered.length + \"件 / 全 \" + RESIDENTS.length + \"件\");\n    setText(el(\"more-note\"),\n      filtered.length > shown\n        ? \"先頭\" + shown + \"件を表示しています。検索で絞り込んでください（残り\" + (filtered.length - shown) + \"件）。\"\n        : \"\");\n  }\n\n  function createResultItem(index) {\n    var resident = RESIDENTS[index];\n    var item = createRow(\"li\", index === selectedIndex ? \"selected\" : \"\");\n    item.dataset.index = String(index);\n    var rid = createRow(\"div\", \"rid\");\n    setText(rid, resident.resident_id);\n    if (!resident.has_coordinates) {\n      var badge = createRow(\"span\", \"badge\");\n      setText(badge, \"座標なし\");\n      rid.appendChild(badge);\n    }\n    var addr = createRow(\"div\", \"addr\");\n    setText(addr, resident.address);\n    item.appendChild(rid);\n    item.appendChild(addr);\n    item.addEventListener(\"click\", function () { select(index); });\n    return item;\n  }\n\n  function updatePosition() {\n    var position = filtered.indexOf(selectedIndex);\n    setText(el(\"position\"), position >= 0 ? (position + 1) + \" / \" + filtered.length : \"\");\n    el(\"prev\").disabled = position <= 0;\n    el(\"next\").disabled = position < 0 || position >= filtered.length - 1;\n  }\n\n  function step(offset) {\n    var position = filtered.indexOf(selectedIndex);\n    if (position < 0) { return; }\n    var next = position + offset;\n    if (next < 0 || next >= filtered.length) { return; }\n    select(filtered[next]);\n  }\n\n  // ---- 選択 ----\n  function select(index) {\n    selectedIndex = index;\n    activeRank = null;\n    Array.prototype.forEach.call(el(\"results\").children, function (node) {\n      node.className = Number(node.dataset.index) === index ? \"selected\" : \"\";\n    });\n    if (index >= 0 && filtered.indexOf(index) < 0) { renderResults(); }\n    updatePosition();\n    renderSelected();\n  }\n\n  function renderSelected() {\n    var summary = el(\"selected-summary\");\n    var candidates = el(\"candidates\");\n    summary.textContent = \"\";\n    candidates.textContent = \"\";\n    if (featureLayer) { featureLayer.clearLayers(); }\n\n    if (selectedIndex < 0) {\n      setText(summary, \"要支援者を選択してください。\");\n      return;\n    }\n\n    var resident = RESIDENTS[selectedIndex];\n    applyRelevantLayers(resident);\n\n    var title = createRow(\"div\", \"title\");\n    setText(title, resident.resident_id);\n    var addr = createRow(\"div\", \"addr\");\n    setText(addr, resident.address);\n    summary.appendChild(title);\n    summary.appendChild(addr);\n\n    var status = createRow(\"div\");\n    setText(status, \"match_status: \" + resident.match_status);\n    summary.appendChild(status);\n\n    if (resident.resident_in_hazard !== null && resident.resident_in_hazard !== undefined) {\n      var hazard = createRow(\"div\", resident.resident_in_hazard ? \"hazard-yes\" : \"hazard-no\");\n      setText(hazard, resident.resident_in_hazard\n        ? \"要支援者地点のハザード: \" + (resident.resident_hazard_types.join(\" / \") || \"該当あり\")\n        : \"要支援者地点のハザード: 区域外\");\n      summary.appendChild(hazard);\n    }\n\n    if (!resident.has_coordinates) {\n      var noCoord = createRow(\"div\", \"empty\");\n      setText(noCoord, \"地図表示できる座標がありません（\" + resident.match_status + \"）。候補も算出されていません。\");\n      summary.appendChild(noCoord);\n    }\n\n    renderCandidateTable(resident);\n    drawResident(resident);\n  }\n\n  function hazardCell(isHazard, types) {\n    var cell = document.createElement(\"td\");\n    if (isHazard === null || isHazard === undefined) {\n      cell.className = \"hazard-no\";\n      setText(cell, \"判定なし\");\n      return cell;\n    }\n    cell.className = isHazard ? \"hazard-yes\" : \"hazard-no\";\n    setText(cell, isHazard ? (types.join(\" / \") || \"該当あり\") : \"なし\");\n    return cell;\n  }\n\n  function renderCandidateTable(resident) {\n    var container = el(\"candidates\");\n    if (!resident.candidates.length) {\n      var empty = createRow(\"div\", \"empty\");\n      setText(empty, \"避難所候補はありません。\");\n      container.appendChild(empty);\n      return;\n    }\n\n    var table = document.createElement(\"table\");\n    var thead = document.createElement(\"thead\");\n    var headRow = document.createElement(\"tr\");\n    [\"順位\", \"避難所名\", \"直線距離\", \"災害種別対応\", \"避難所地点のハザード\", \"直線交差ハザード\"]\n      .forEach(function (name) {\n        var th = document.createElement(\"th\");\n        setText(th, name);\n        headRow.appendChild(th);\n      });\n    thead.appendChild(headRow);\n    table.appendChild(thead);\n\n    var tbody = document.createElement(\"tbody\");\n    resident.candidates.forEach(function (candidate) {\n      var row = document.createElement(\"tr\");\n      row.dataset.rank = String(candidate.rank);\n      if (activeRank === candidate.rank) { row.className = \"active\"; }\n\n      var rankCell = document.createElement(\"td\");\n      var chip = createRow(\"span\", \"rank-chip\");\n      chip.style.background = RANK_COLORS[candidate.rank - 1] || \"#555\";\n      setText(chip, candidate.rank);\n      rankCell.appendChild(chip);\n      var rankLabel = document.createElement(\"span\");\n      setText(rankLabel, \" 候補\" + candidate.rank);\n      rankCell.appendChild(rankLabel);\n      row.appendChild(rankCell);\n\n      var nameCell = document.createElement(\"td\");\n      setText(nameCell, candidate.name);\n      row.appendChild(nameCell);\n\n      var distanceCell = document.createElement(\"td\");\n      setText(distanceCell, candidate.distance_m == null ? \"\" : candidate.distance_m + \" m\");\n      row.appendChild(distanceCell);\n\n      var supportCell = document.createElement(\"td\");\n      setText(supportCell, candidate.disaster_support || \"\");\n      row.appendChild(supportCell);\n\n      row.appendChild(hazardCell(candidate.shelter_in_hazard, candidate.shelter_hazard_types));\n      row.appendChild(hazardCell(candidate.straight_line_intersects_hazard, candidate.straight_line_hazard_types));\n\n      row.addEventListener(\"click\", function () {\n        activeRank = activeRank === candidate.rank ? null : candidate.rank;\n        el(\"candidates\").textContent = \"\";\n        renderCandidateTable(resident);\n        drawResident(resident);\n      });\n      tbody.appendChild(row);\n    });\n    table.appendChild(tbody);\n    container.appendChild(table);\n\n    var note = createRow(\"div\", \"count\");\n    setText(note, \"距離は直線距離（geodesic）です。道路距離ではありません。順位はハザード情報では変わりません。\");\n    container.appendChild(note);\n  }\n\n  // ---- 地図描画（選択した1人だけ） ----\n  function drawResident(resident) {\n    if (!map || !featureLayer) { return; }\n    featureLayer.clearLayers();\n    if (!resident.has_coordinates) { return; }\n\n    var residentLatLng = [resident.latitude, resident.longitude];\n    var points = [residentLatLng];\n\n    resident.candidates.forEach(function (candidate) {\n      var candidateLatLng = [candidate.latitude, candidate.longitude];\n      points.push(candidateLatLng);\n      var color = RANK_COLORS[candidate.rank - 1] || \"#555\";\n      var emphasised = activeRank === candidate.rank;\n\n      L.polyline([residentLatLng, candidateLatLng], {\n        color: color,\n        weight: emphasised ? 5 : 3,\n        opacity: activeRank && !emphasised ? 0.35 : 0.9,\n        dashArray: \"6 5\"\n      }).addTo(featureLayer).bindTooltip(\n        textTooltip(\"候補\" + candidate.rank + \"への直線（避難経路ではありません）\"));\n\n      L.marker(candidateLatLng, {\n        icon: L.divIcon({\n          className: \"\",\n          // ここで組み立てるのは候補番号（このNotebookが振る整数）と配色だけで、\n          // 利用者データは入れない\n          html: '<div class=\"marker-pin\" style=\"background:' + color + ';\">' + candidate.rank + '</div>',\n          iconSize: [22, 22],\n          iconAnchor: [11, 11]\n        }),\n        opacity: activeRank && !emphasised ? 0.5 : 1\n      }).addTo(featureLayer).bindTooltip(\n        textTooltip(\"候補\" + candidate.rank + \": \" + candidate.name));\n    });\n\n    L.circleMarker(residentLatLng, {\n      radius: 9, color: \"#fff\", weight: 2, fillColor: RESIDENT_COLOR, fillOpacity: 1\n    }).addTo(featureLayer).bindTooltip(textTooltip(\"要支援者地点\"));\n\n    try {\n      // 候補詳細の高さが変わると地図の表示サイズも変わるため、\n      // 収める範囲を計算する前にLeafletへ現在の大きさを取り直させる\n      map.invalidateSize({ animate: false });\n      map.fitBounds(L.latLngBounds(points).pad(0.25));\n    } catch (error) {\n      map.setView(residentLatLng, 15);\n    }\n  }\n\n  // ---- 起動 ----\n  setText(el(\"generated-at\"), DATA.generated_at ? \"（作成: \" + DATA.generated_at + \"）\" : \"\");\n  initMap();\n  renderLayerControls();\n  el(\"search\").addEventListener(\"input\", applyFilter);\n  el(\"prev\").addEventListener(\"click\", function () { step(-1); });\n  el(\"next\").addEventListener(\"click\", function () { step(1); });\n  el(\"basemap-toggle\").addEventListener(\"change\", function (event) {\n    setBasemapEnabled(event.target.checked);\n  });\n  el(\"layers-all\").addEventListener(\"click\", function () { setAllLayers(true); });\n  el(\"layers-none\").addEventListener(\"click\", function () { setAllLayers(false); });\n  applyFilter();\n})();\n</script>\n</body>\n</html>\n"

print("レビュー用HTMLの生成処理を定義しました。")


In [ ]:
# ===== 結果確認・CSV出力・レビュー用HTML出力 =====
# CSVを出力する前に、Notebook上で処理結果の概要と先頭数行を確認します。
# 結果は Excelで文字化けしにくい utf-8-sig（UTF-8 BOM付き）でCSVに出力し、ブラウザへダウンロードします。
#
# 続けて、結果を地図上で目視確認するためのレビュー用HTMLを sheltermatch_review.zip として
# 出力します。ZIPを展開して review.html を開くと、要支援者を1人ずつ選んで候補・距離・ハザードを
# 地図で確認できます。正式なデータ成果物は assigned_shelters.csv で、HTMLはその確認用の
# 補助成果物です（避難所の割り当て結果ではありません）。
#
# 出力CSV・HTMLには個人情報が含まれ得るため、取り扱いに注意してください。

_stage_start = time.perf_counter()

total_residents = len(final_df)
ok_count = int((final_df["match_status"] == "ok").sum())
no_coord_count = int((final_df["match_status"] == "no_coordinates").sum())
invalid_coord_count = int((final_df["match_status"] == "invalid_coordinates").sum())

print(f"要支援者件数: {total_residents}件")
print(f"距離計算できた件数: {ok_count}件")
print(f"座標が空欄のため距離計算できなかった件数: {no_coord_count}件")
print(f"座標が範囲外で不正なため距離計算できなかった件数: {invalid_coord_count}件")
print(f"距離計算に使用した有効な避難所件数: {len(shelters_valid)}件")

if ENABLE_HAZARD_CHECK:
    def count_true(column_name_format):
        """候補1〜TOP_Nの指定列でTrueだった件数を合計する（延べ件数）。"""
        return sum(
            int(final_df[column_name_format.format(n=n)].eq(True).sum())
            for n in range(1, TOP_N + 1)
        )

    resident_hazard_count = int(final_df["resident_in_hazard"].eq(True).sum())
    shelter_hazard_count = count_true("candidate_{n}_shelter_in_hazard")
    line_hazard_count = count_true("candidate_{n}_straight_line_intersects_hazard")

    print(f"ハザード区域内（境界上含む）にいる要支援者数: {resident_hazard_count}件")
    print(f"ハザード区域内にある候補避難所の件数（延べ、TOP_N分の合計）: {shelter_hazard_count}件")
    print(f"候補避難所への直線がハザード区域と交差する件数（延べ、TOP_N分の合計）: {line_hazard_count}件")

display(final_df.head())

OUTPUT_FILENAME = "assigned_shelters.csv"

final_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
print(f"'{OUTPUT_FILENAME}' を出力しました。")

# 正式なデータ成果物である結果CSVは、レビュー用HTMLの作成結果に関わらず受け取れるよう
# 先にダウンロードする。
files.download(OUTPUT_FILENAME)

# レビュー用HTMLは結果CSVと同じ実行結果から作る（CSVを読み直して独自解釈はしない）。
# 内容がCSVと食い違う場合は、build_review_package の中で処理を止める。
review_zip_path = build_review_package(
    final_df, review_rows, hazard_gdf, shelters_valid, TOP_N, output_dir=Path(".")
)
print(f"'{review_zip_path.name}' を出力しました（展開して review.html を開いてください）。")

TIMINGS["output"] = time.perf_counter() - _stage_start
print(f"[処理時間] 結果CSV・レビューHTML生成: {TIMINGS['output']:.2f}秒")

total_time = sum(TIMINGS.values())
print("\n処理時間:")
print(f"  要支援者CSV読込        {TIMINGS.get('residents_csv', 0):.1f}秒")
print(f"  避難所データ準備        {TIMINGS.get('shelters', 0):.1f}秒")
print(f"  ハザードデータ読込      {TIMINGS.get('hazard', 0):.1f}秒")
print(f"  候補・ハザード判定      {TIMINGS.get('candidates', 0):.1f}秒")
print(f"  CSV・HTML出力          {TIMINGS.get('output', 0):.1f}秒")
print(f"  合計                   {total_time:.1f}秒")

files.download(str(review_zip_path))
